# Toolset Reliability Sensitivity Ablation (Legacy Adapter Study)

This notebook is retained for historical adapter sensitivity studies.

Use the core CLI workflow for the repository's primary goals.
Use this notebook only when reproducing adapter-specific ablation experiments.

## Historical intent
- Check whether adapter variants change tool-calling behavior on target toolsets
- Stress-test sensitivity settings under stronger schema and timeout pressure
- Produce reproducible artifacts for archival comparison

In [39]:
!nvidia-smi

Fri Apr  3 11:32:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import time
from pathlib import Path
from getpass import getpass

REPO_NAME = 'tool-calling-reliability-benchmark'
REPO_URL = 'https://github.com/aaliyan1230/tool-calling-reliability-benchmark.git'

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    kaggle_repo = Path('/kaggle/working') / REPO_NAME
    if not kaggle_repo.exists():
        print(f'[setup] Cloning repo to {kaggle_repo} ...')
        subprocess.run(['git', 'clone', REPO_URL, str(kaggle_repo)], check=True)
    repo_root = kaggle_repo

REPO_ROOT = repo_root.resolve()
os.chdir(REPO_ROOT)

if shutil.which('uv') is None:
    print('[setup] Installing uv ...')
    subprocess.run(['python', '-m', 'pip', 'install', '-q', 'uv'], check=True)

print('Repo root:', REPO_ROOT)
print('Kernel cwd:', Path.cwd())

[setup] Cloning repo to /kaggle/working/tool-calling-reliability-benchmark ...


Cloning into '/kaggle/working/tool-calling-reliability-benchmark'...


Repo root: /kaggle/working/tool-calling-reliability-benchmark
Kernel cwd: /kaggle/working/tool-calling-reliability-benchmark


In [7]:
HF_TOKEN = str(os.environ.get('HF_TOKEN', '')).strip()
if not HF_TOKEN:
    HF_TOKEN = getpass('Enter HF_TOKEN (input hidden): ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required.')
os.environ['HF_TOKEN'] = HF_TOKEN

KAGGLE_USERNAME = str(os.environ.get('KAGGLE_USERNAME', '')).strip()
if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = input('Enter KAGGLE_USERNAME: ').strip()

KAGGLE_KEY = str(os.environ.get('KAGGLE_KEY', '')).strip()
if not KAGGLE_KEY:
    KAGGLE_KEY = str(os.environ.get('KAGGLE_API_TOKEN', '')).strip()
if not KAGGLE_KEY:
    KAGGLE_KEY = getpass('Enter KAGGLE_KEY (input hidden): ').strip()

if not KAGGLE_USERNAME or not KAGGLE_KEY:
    raise RuntimeError('KAGGLE_USERNAME and KAGGLE_KEY are required.')

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

try:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('[auth] Hugging Face login succeeded.')
except Exception as exc:
    print('[auth] HF login warning:', exc)

print('[auth] Kaggle credentials configured for runtime.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[auth] Hugging Face login succeeded.
[auth] Kaggle credentials configured for runtime.


In [8]:
LABEL_PREFIX = 'toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1'
SEEDS = '11,22,33,44,55,66,77,88,99,111'
BASE_PLANNER = 'configs/planners/hf_qwen2_5_3b_base.json'
FT_PLANNER = 'configs/planners/hf_qwen2_5_3b_ft.json'
BASE_CONFIG = REPO_ROOT / 'configs' / 'baseline.json'
SENSITIVITY_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_sensitivity_v1.json'

ADAPTER_DATASET = 'aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts'
PUBLISH_DATASET_SLUG = 'tcrb-qwen25-3b-toolset-reliability-sensitivity-v1'
PUBLISH_DATASET_TITLE = 'TCRB Qwen2.5-3B Toolset Reliability Sensitivity V1'

print('LABEL_PREFIX =', LABEL_PREFIX)
print('SEEDS =', SEEDS)
print('BASE_CONFIG =', BASE_CONFIG)
print('SENSITIVITY_CONFIG =', SENSITIVITY_CONFIG)

LABEL_PREFIX = toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1
SEEDS = 11,22,33,44,55,66,77,88,99,111
BASE_CONFIG = /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json
SENSITIVITY_CONFIG = /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_sensitivity_v1.json


In [9]:
cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']
faults = dict(cfg.get('fault_probabilities', {}))
faults['malformed_schema'] = max(float(faults.get('malformed_schema', 0.06)), 0.12)
faults['timeout'] = max(float(faults.get('timeout', 0.08)), 0.12)
cfg['fault_probabilities'] = faults
cfg['max_attempts'] = max(int(cfg.get('max_attempts', 4)), 5)
cfg['time_budget_ms'] = max(int(cfg.get('time_budget_ms', 1800)), 2200)
SENSITIVITY_CONFIG.parent.mkdir(parents=True, exist_ok=True)
SENSITIVITY_CONFIG.write_text(json.dumps(cfg, indent=2) + '\n', encoding='utf-8')
print('Wrote sensitivity config:', SENSITIVITY_CONFIG)

Wrote sensitivity config: /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_sensitivity_v1.json


In [10]:
pull_script = REPO_ROOT / 'scripts' / 'pull_kaggle_adapter.py'
pull_cmd = ['uv', 'run', 'python', str(pull_script), '--dataset', ADAPTER_DATASET, '--repo-root', '.']
print('Running:', ' '.join(pull_cmd))
res = subprocess.run(pull_cmd, text=True, capture_output=True, check=False)
if res.stdout:
    print(res.stdout)
if res.returncode != 0:
    if res.stderr:
        print(res.stderr)
    raise RuntimeError(f'Adapter pull failed with code {res.returncode}')

Running: uv run python /kaggle/working/tool-calling-reliability-benchmark/scripts/pull_kaggle_adapter.py --dataset aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts --repo-root .
Dataset URL: https://www.kaggle.com/datasets/aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts
License(s): CC0-1.0

{
  "dataset": "aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts",
  "download_dir": "/kaggle/working/tool-calling-reliability-benchmark/tmp/kaggle_adapter_pull",
  "adapter_root": "/kaggle/working/tool-calling-reliability-benchmark/tmp/kaggle_adapter_pull/adapter",
  "target_dir": "/kaggle/working/tool-calling-reliability-benchmark/outputs/ft-notebook/final"
}
Adapter materialized successfully.



In [ ]:
required_mods = ['torch', 'transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes', 'wrapt']

probe_cmd = [

    'uv', 'run', 'python', '-c',

    "import importlib.util as u; mods=%r; missing=[m for m in mods if u.find_spec(m) is None]; print('MISSING=' + ','.join(missing))" % required_mods,

]

probe = subprocess.run(probe_cmd, text=True, capture_output=True, check=False)

probe_out = (probe.stdout or '').strip()

print('[deps] Probe output:', probe_out)


markdown
fastsig-md-01
## Fast-Signal Implementation Path (24h, Low Compute)

This compact path is the primary implementation target for the current cycle.

Execution intent:
- Run one model-sensitive enriched triad (base vs finetuned vs null-adapter)
- Confirm non-flat signal and adapter advantage over null control
- Run one lightweight northstar check using the same setup

Fast-signal acceptance criteria (pre-registered):
- mean success delta (ft-base) > 0.0
- mean invalid delta (ft-base) <= 0.0
- adapter advantage vs null on success > 0.003
- adapter advantage vs null on invalid <= 0.0
- non-flat check: max absolute primary delta > 1e-4
code

fastsig-code-setup
import json
import subprocess
from pathlib import Path

FAST_WORKLOAD = 'workloads/enriched/customer_support.json'
FAST_SEEDS = '11,22,33'
FAST_LABEL_PREFIX = f"{LABEL_PREFIX}-fastsig-cs-v1"
FAST_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_fast_signal_cs_v1.json'
FAST_NULL_PLANNER = REPO_ROOT / 'configs' / 'planners' / 'hf_qwen2_5_3b_ft_nulladapter.json'

fast_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
fast_cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']
fast_faults = dict(fast_cfg.get('fault_probabilities', {}))
fast_faults['malformed_schema'] = max(float(fast_faults.get('malformed_schema', 0.06)), 0.10)
fast_faults['timeout'] = max(float(fast_faults.get('timeout', 0.08)), 0.10)
fast_cfg['fault_probabilities'] = fast_faults
fast_cfg['max_attempts'] = max(int(fast_cfg.get('max_attempts', 4)), 5)
fast_cfg['time_budget_ms'] = max(int(fast_cfg.get('time_budget_ms', 1800)), 2200)

FAST_CONFIG.write_text(json.dumps(fast_cfg, indent=2) + '\n', encoding='utf-8')

if not FAST_NULL_PLANNER.exists():
    null_obj = {
        'type': 'hf_local',
        'name': 'hf_qwen2_5_3b_ft_nulladapter',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
    }
    FAST_NULL_PLANNER.write_text(json.dumps(null_obj, indent=2) + '\n', encoding='utf-8')

print('FAST_CONFIG =', FAST_CONFIG)
print('FAST_WORKLOAD =', FAST_WORKLOAD)
print('FAST_SEEDS =', FAST_SEEDS)
print('FAST policies =', fast_cfg['policies'])
print('FAST faults =', fast_cfg['fault_probabilities'])
print('FAST_NULL_PLANNER =', FAST_NULL_PLANNER)
code

fastsig-code-triad
def run_multi_seed_fast(workload: str, planner_cfg: str, label: str) -> None:
    cmd = [
        'uv', 'run', 'tcrb', 'multi-seed',
        '--config', str(FAST_CONFIG),
        '--workload', workload,
        '--seeds', FAST_SEEDS,
        '--planner-config', planner_cfg,
        '--label', label,
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'fast multi-seed failed for {label} with code {rc}')


def load_policy_means_fast(label: str) -> dict:
    payload_path = REPO_ROOT / 'runs' / label / 'multi_seed.json'
    payload = json.loads(payload_path.read_text(encoding='utf-8'))
    return {r['policy']: r['metrics'] for r in payload.get('aggregate_policy_metrics', [])}


fast_base_label = f'{FAST_LABEL_PREFIX}-base'
fast_ft_label = f'{FAST_LABEL_PREFIX}-ft'
fast_null_label = f'{FAST_LABEL_PREFIX}-null'

run_multi_seed_fast(FAST_WORKLOAD, BASE_PLANNER, fast_base_label)
run_multi_seed_fast(FAST_WORKLOAD, FT_PLANNER, fast_ft_label)
run_multi_seed_fast(FAST_WORKLOAD, str(FAST_NULL_PLANNER.relative_to(REPO_ROOT)), fast_null_label)

fast_base = load_policy_means_fast(fast_base_label)
fast_ft = load_policy_means_fast(fast_ft_label)
fast_null = load_policy_means_fast(fast_null_label)
fast_policies = sorted(set(fast_base) & set(fast_ft) & set(fast_null))

ft_s = [fast_ft[p]['task_success_rate']['mean'] - fast_base[p]['task_success_rate']['mean'] for p in fast_policies]
ft_i = [fast_ft[p]['invalid_tool_call_rate']['mean'] - fast_base[p]['invalid_tool_call_rate']['mean'] for p in fast_policies]
null_s = [fast_null[p]['task_success_rate']['mean'] - fast_base[p]['task_success_rate']['mean'] for p in fast_policies]
null_i = [fast_null[p]['invalid_tool_call_rate']['mean'] - fast_base[p]['invalid_tool_call_rate']['mean'] for p in fast_policies]

fast_mean_success_delta = sum(ft_s) / len(ft_s) if ft_s else 0.0
fast_mean_invalid_delta = sum(ft_i) / len(ft_i) if ft_i else 0.0
fast_null_success_delta = sum(null_s) / len(null_s) if null_s else 0.0
fast_null_invalid_delta = sum(null_i) / len(null_i) if null_i else 0.0

fast_adapter_adv_success = fast_mean_success_delta - fast_null_success_delta
fast_adapter_adv_invalid = fast_mean_invalid_delta - fast_null_invalid_delta
fast_nonflat = max(
    abs(fast_mean_success_delta),
    abs(fast_mean_invalid_delta),
    abs(fast_adapter_adv_success),
    abs(fast_adapter_adv_invalid),
) > 1e-4

fast_signal_pass = (
    (fast_mean_success_delta > 0.0)
    and (fast_mean_invalid_delta <= 0.0)
    and (fast_adapter_adv_success > 0.003)
    and (fast_adapter_adv_invalid <= 0.0)
    and fast_nonflat
)

FAST_TRIAD_SUMMARY = {
    'label_prefix': FAST_LABEL_PREFIX,
    'workload': FAST_WORKLOAD,
    'seeds': FAST_SEEDS,
    'policies': fast_policies,
    'ft_base_success_delta': fast_mean_success_delta,
    'ft_base_invalid_delta': fast_mean_invalid_delta,
    'null_base_success_delta': fast_null_success_delta,
    'null_base_invalid_delta': fast_null_invalid_delta,
    'adapter_adv_success': fast_adapter_adv_success,
    'adapter_adv_invalid': fast_adapter_adv_invalid,
    'nonflat': fast_nonflat,
    'fast_signal_pass': fast_signal_pass,
}

print('\n=== Fast Triad Summary ===')
print(json.dumps(FAST_TRIAD_SUMMARY, indent=2))
code

fastsig-code-ns
FAST_NS_LABEL_PREFIX = f'{FAST_LABEL_PREFIX}-northstar'

northstar_cmd = [
    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',
    '--config', str(FAST_CONFIG),
    '--workload', FAST_WORKLOAD,
    '--seeds', FAST_SEEDS,
    '--base-planner-config', BASE_PLANNER,
    '--ft-planner-config', FT_PLANNER,
    '--label-prefix', FAST_NS_LABEL_PREFIX,
]

print('Running:', ' '.join(northstar_cmd))
ns_proc = subprocess.Popen(northstar_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert ns_proc.stdout is not None
for line in ns_proc.stdout:
    print(line, end='')
ns_rc = ns_proc.wait()
if ns_rc != 0:
    raise RuntimeError(f'fast northstar run failed with code {ns_rc}')

run_root = REPO_ROOT / 'runs'
ns_base_ms = run_root / f'{FAST_NS_LABEL_PREFIX}-base-ms' / 'multi_seed.json'
ns_ft_ms = run_root / f'{FAST_NS_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'
ns_matrix_json = run_root / f'{FAST_NS_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [ns_base_ms, ns_ft_ms, ns_matrix_json]:
    print('-', p, 'exists=' + str(p.exists()))

ns_gate_dir = run_root / f'{FAST_NS_LABEL_PREFIX}-study-gate'
ns_gate_dir.mkdir(parents=True, exist_ok=True)
ns_gate_json = ns_gate_dir / 'study_gate.json'
ns_gate_md = ns_gate_dir / 'study_gate.md'

gate_cmd = [
    'uv', 'run', 'python', '-m', 'tcrb', 'study-gate',
    '--base-run', str(ns_base_ms),
    '--finetuned-run', str(ns_ft_ms),
    '--matrix-json', str(ns_matrix_json),
    '--require-matrix-signal',
    '--require-matrix-not-fail',
    '--output-json', str(ns_gate_json),
    '--output-report', str(ns_gate_md),
]

print('Running:', ' '.join(gate_cmd))
gate_res = subprocess.run(gate_cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)
if gate_res.stdout:
    print(gate_res.stdout)
if gate_res.returncode != 0 and gate_res.stderr:
    print(gate_res.stderr)

ns_matrix_obj = json.loads(ns_matrix_json.read_text(encoding='utf-8'))
ns_gate_obj = json.loads(ns_gate_json.read_text(encoding='utf-8')) if ns_gate_json.exists() else {}

ns_rows = ns_matrix_obj.get('rows', [])
ns_max_abs_matrix_delta = max(
    [
        abs(float(r.get('delta_first_tool_accuracy', 0.0)))
        for r in ns_rows
    ]
    + [
        abs(float(r.get('delta_sequence_prefix_accuracy', 0.0)))
        for r in ns_rows
    ]
    + [0.0]
)

ns_study_verdict = str(ns_gate_obj.get('verdict', 'MISSING'))
ns_matrix_verdict = str(ns_matrix_obj.get('portfolio_verdict', 'MISSING'))
if ns_study_verdict == 'PASS':
    ns_fail_class = 'PASS'
elif ns_max_abs_matrix_delta <= 1e-4:
    ns_fail_class = 'STRUCTURAL_FLAT_FAIL'
else:
    ns_fail_class = 'THRESHOLD_TIGHT_FAIL'

FAST_NORTHSTAR_SUMMARY = {
    'label_prefix': FAST_NS_LABEL_PREFIX,
    'matrix_portfolio_verdict': ns_matrix_verdict,
    'study_gate_verdict': ns_study_verdict,
    'matrix_max_abs_delta': ns_max_abs_matrix_delta,
    'failure_classification': ns_fail_class,
    'gate_checks': ns_gate_obj.get('checks', []),
}

print('\n=== Fast Northstar Summary ===')
print(json.dumps(FAST_NORTHSTAR_SUMMARY, indent=2))
code

fastsig-code-decision
print('=== Fast Signal Final Decision ===')
print('triad_fast_signal_pass =', FAST_TRIAD_SUMMARY.get('fast_signal_pass'))
print('northstar_failure_classification =', FAST_NORTHSTAR_SUMMARY.get('failure_classification'))

if FAST_TRIAD_SUMMARY.get('fast_signal_pass'):
    if FAST_NORTHSTAR_SUMMARY.get('failure_classification') == 'PASS':
        decision = 'GO: meaningful signal found and northstar-lite passed.'
    else:
        decision = 'GO-PHASE-2: meaningful signal found; proceed to northstar-grade portfolio run.'
else:
    if FAST_NORTHSTAR_SUMMARY.get('failure_classification') == 'STRUCTURAL_FLAT_FAIL':
        decision = 'PIVOT: benchmark still flat; refresh finetune data before more benchmarking.'
    else:
        decision = 'HOLD: weak signal; run one targeted hard-slice triad before retraining.'

print('decision =', decision)

FAST_DECISION_SUMMARY = {
    'triad': FAST_TRIAD_SUMMARY,
    'northstar': FAST_NORTHSTAR_SUMMARY,
    'decision': decision,
}
print('\nFAST_DECISION_SUMMARY_JSON')
print(json.dumps(FAST_DECISION_SUMMARY, indent=2))
code
#VSC-068f0898
python
cmd = [
    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',
    '--config', str(SENSITIVITY_CONFIG),
    '--seeds', SEEDS,
    '--base-planner-config', BASE_PLANNER,
    '--ft-planner-config', FT_PLANNER,
    '--label-prefix', LABEL_PREFIX,
]
print('Running:', ' '.join(cmd))
started = time.time()
proc = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'[toolset-reliability] Total elapsed: {time.time() - started:.1f}s')
if rc != 0:
    raise RuntimeError(f'Study run failed with code {rc}')
code
#VSC-1f32bae2
python
run_root = REPO_ROOT / 'runs'
base_ms = run_root / f'{LABEL_PREFIX}-base-ms' / 'multi_seed.json'
ft_ms = run_root / f'{LABEL_PREFIX}-ft-ms' / 'multi_seed.json'
matrix_json = run_root / f'{LABEL_PREFIX}-matrix' / 'matrix.json'
for p in [base_ms, ft_ms, matrix_json]:
    print('-', p, 'exists=' + str(p.exists()))
b = json.loads(base_ms.read_text(encoding='utf-8'))
f = json.loads(ft_ms.read_text(encoding='utf-8'))
m = json.loads(matrix_json.read_text(encoding='utf-8'))
by_b = {r['policy']: r['metrics'] for r in b.get('aggregate_policy_metrics', [])}
by_f = {r['policy']: r['metrics'] for r in f.get('aggregate_policy_metrics', [])}
policies = sorted(set(by_b) & set(by_f))
succ = [by_f[p]['task_success_rate']['mean'] - by_b[p]['task_success_rate']['mean'] for p in policies]
inv = [by_f[p]['invalid_tool_call_rate']['mean'] - by_b[p]['invalid_tool_call_rate']['mean'] for p in policies]
mean_success_delta = sum(succ) / len(succ) if succ else 0.0
mean_invalid_delta = sum(inv) / len(inv) if inv else 0.0
print('=== Toolset Reliability Summary ===')
print('policies:', policies)
print(f'mean success delta (ft-base): {mean_success_delta:+.4f}')
print(f'mean invalid delta (ft-base): {mean_invalid_delta:+.4f}')
print('matrix portfolio verdict:', m.get('portfolio_verdict'))
code
#VSC-bd6cc78b
python
from pathlib import Path

import json
import subprocess


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate

    fallbacks = [
        Path('/kaggle/working/tool-calling-reliability-benchmark'),
        Path('/Users/aaliyan/aaliyan/tool-calling-reliability-benchmark'),
    ]
    for candidate in fallbacks:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    return None


repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    raise RuntimeError('Could not locate repository root for study-gate run.')
REPO_ROOT = repo_root.resolve()

try:
    LABEL_PREFIX
except NameError:
    LABEL_PREFIX = 'toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1'

run_root = REPO_ROOT / 'runs'
base_ms = run_root / f'{LABEL_PREFIX}-base-ms' / 'multi_seed.json'
ft_ms = run_root / f'{LABEL_PREFIX}-ft-ms' / 'multi_seed.json'
matrix_json = run_root / f'{LABEL_PREFIX}-matrix' / 'matrix.json'
gate_dir = run_root / f'{LABEL_PREFIX}-study-gate'
gate_dir.mkdir(parents=True, exist_ok=True)
gate_json = gate_dir / 'study_gate.json'
gate_md = gate_dir / 'study_gate.md'

cmd = [
    'uv', 'run', 'python', '-m', 'tcrb', 'study-gate',
    '--base-run', str(base_ms),
    '--finetuned-run', str(ft_ms),
    '--matrix-json', str(matrix_json),
    '--require-matrix-signal',
    '--require-matrix-not-fail',
    '--output-json', str(gate_json),
    '--output-report', str(gate_md),
]

print('Repo root:', REPO_ROOT)
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)
if res.stdout:
    print(res.stdout)
if res.returncode != 0 and res.stderr:
    print(res.stderr)
print('study-gate exit code =', res.returncode)

if gate_json.exists():
    payload = json.loads(gate_json.read_text(encoding='utf-8'))
    print('study-gate verdict =', payload.get('verdict'))
    for check in payload.get('checks', []):
        print('-', check.get('name'), 'PASS' if check.get('passed') else 'FAIL', 'value=', check.get('value'), 'threshold=', check.get('threshold'))
    print('study-gate markdown =', gate_md)
else:
    print('study-gate JSON was not generated:', gate_json)
markdown
#VSC-1f802332
markdown
## Calibrated Recovery Arm



This arm reduces stress relative to the sensitivity run and widens policy coverage to seek a practical, favorable outcome in the same runtime.

code
#VSC-6e2c0392
python
RECOVERY_LABEL_PREFIX = f"{LABEL_PREFIX}-recovery"

RECOVERY_SEEDS = '11,22,33,44,55'

RECOVERY_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_recovery_v1.json'



recovery_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

recovery_cfg['policies'] = [

    'naive_retry',

    'exponential_backoff_jitter',

    'majority_vote',

    'self_consistency',

]

recovery_faults = dict(recovery_cfg.get('fault_probabilities', {}))

recovery_faults['malformed_schema'] = min(float(recovery_faults.get('malformed_schema', 0.06)), 0.06)

recovery_faults['timeout'] = min(float(recovery_faults.get('timeout', 0.08)), 0.08)

recovery_cfg['fault_probabilities'] = recovery_faults

recovery_cfg['max_attempts'] = max(int(recovery_cfg.get('max_attempts', 4)), 5)

recovery_cfg['time_budget_ms'] = max(int(recovery_cfg.get('time_budget_ms', 1800)), 2200)



RECOVERY_CONFIG.write_text(json.dumps(recovery_cfg, indent=2) + '\n', encoding='utf-8')

print('Wrote recovery config:', RECOVERY_CONFIG)

print('Recovery policies:', recovery_cfg['policies'])

print('Recovery faults:', recovery_cfg['fault_probabilities'])



recovery_cmd = [

    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

    '--config', str(RECOVERY_CONFIG),

    '--seeds', RECOVERY_SEEDS,

    '--base-planner-config', BASE_PLANNER,

    '--ft-planner-config', FT_PLANNER,

    '--label-prefix', RECOVERY_LABEL_PREFIX,

]

print('Running:', ' '.join(recovery_cmd))

started_recovery = time.time()

recovery_proc = subprocess.Popen(

    recovery_cmd,

    text=True,

    cwd=str(REPO_ROOT),

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

)

assert recovery_proc.stdout is not None

for line in recovery_proc.stdout:

    print(line, end='')

recovery_rc = recovery_proc.wait()

print(f'[recovery] Total elapsed: {time.time() - started_recovery:.1f}s')

if recovery_rc != 0:

    raise RuntimeError(f'Recovery run failed with code {recovery_rc}')



recovery_root = REPO_ROOT / 'runs'

recovery_base = recovery_root / f'{RECOVERY_LABEL_PREFIX}-base-ms' / 'multi_seed.json'

recovery_ft = recovery_root / f'{RECOVERY_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'

recovery_matrix = recovery_root / f'{RECOVERY_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [recovery_base, recovery_ft, recovery_matrix]:

    print('-', p, 'exists=' + str(p.exists()))



rb = json.loads(recovery_base.read_text(encoding='utf-8'))

rf = json.loads(recovery_ft.read_text(encoding='utf-8'))

rm = json.loads(recovery_matrix.read_text(encoding='utf-8'))



rb_by = {r['policy']: r['metrics'] for r in rb.get('aggregate_policy_metrics', [])}

rf_by = {r['policy']: r['metrics'] for r in rf.get('aggregate_policy_metrics', [])}

recovery_policies = sorted(set(rb_by) & set(rf_by))

recovery_succ = [rf_by[p]['task_success_rate']['mean'] - rb_by[p]['task_success_rate']['mean'] for p in recovery_policies]

recovery_inv = [rf_by[p]['invalid_tool_call_rate']['mean'] - rb_by[p]['invalid_tool_call_rate']['mean'] for p in recovery_policies]

recovery_mean_success_delta = sum(recovery_succ) / len(recovery_succ) if recovery_succ else 0.0

recovery_mean_invalid_delta = sum(recovery_inv) / len(recovery_inv) if recovery_inv else 0.0

recovery_desirable = (recovery_mean_success_delta >= 0.0) and (recovery_mean_invalid_delta <= 0.0)



print('\n=== Recovery Arm Summary ===')

print('policies:', recovery_policies)

print(f'mean success delta (ft-base): {recovery_mean_success_delta:+.4f}')

print(f'mean invalid delta (ft-base): {recovery_mean_invalid_delta:+.4f}')

print('matrix portfolio verdict:', rm.get('portfolio_verdict'))

print('desirable outcome hit:', recovery_desirable)
code
#VSC-dd2c7fd6
python
print('RECOVERY_LABEL_PREFIX =', RECOVERY_LABEL_PREFIX)

print(f'recovery mean success delta (ft-base): {recovery_mean_success_delta:+.4f}')

print(f'recovery mean invalid delta (ft-base): {recovery_mean_invalid_delta:+.4f}')

print('recovery matrix portfolio verdict:', rm.get('portfolio_verdict'))

print('desirable outcome hit:', recovery_desirable)
markdown
#VSC-13a54db5
markdown
## Comparator Calibration Arm (Fast Sanity)



This arm uses tiny comparator planners to establish a concrete pass-oriented sanity signal in the same runtime.

code
#VSC-00ef34c7
python
CAL_LABEL_PREFIX = 'toolsetrel-hf-qwen25-3b-calibration-v1'

CAL_SEEDS = '11,22,33,44,55'

CAL_BASE_PLANNER = 'configs/planners/hf_qwen2_5_3b_base.json'

CAL_FT_PLANNER = 'configs/planners/hf_qwen2_5_3b_ft.json'



cal_cmd = [

    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

    '--config', str(BASE_CONFIG),

    '--seeds', CAL_SEEDS,

    '--base-planner-config', CAL_BASE_PLANNER,

    '--ft-planner-config', CAL_FT_PLANNER,

    '--label-prefix', CAL_LABEL_PREFIX,

]

print('Running:', ' '.join(cal_cmd))

started_cal = time.time()

cal_proc = subprocess.Popen(cal_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

assert cal_proc.stdout is not None

for line in cal_proc.stdout:

    print(line, end='')

cal_rc = cal_proc.wait()

print(f'[calibration] Total elapsed: {time.time() - started_cal:.1f}s')

if cal_rc != 0:

    raise RuntimeError(f'Calibration run failed with code {cal_rc}')



cal_root = REPO_ROOT / 'runs'

cal_base = cal_root / f'{CAL_LABEL_PREFIX}-base-ms' / 'multi_seed.json'

cal_ft = cal_root / f'{CAL_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'

cal_matrix = cal_root / f'{CAL_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [cal_base, cal_ft, cal_matrix]:

    print('-', p, 'exists=' + str(p.exists()))



cb = json.loads(cal_base.read_text(encoding='utf-8'))

cf = json.loads(cal_ft.read_text(encoding='utf-8'))

cm = json.loads(cal_matrix.read_text(encoding='utf-8'))

cb_by = {r['policy']: r['metrics'] for r in cb.get('aggregate_policy_metrics', [])}

cf_by = {r['policy']: r['metrics'] for r in cf.get('aggregate_policy_metrics', [])}

cal_policies = sorted(set(cb_by) & set(cf_by))

cal_succ = [cf_by[p]['task_success_rate']['mean'] - cb_by[p]['task_success_rate']['mean'] for p in cal_policies]

cal_inv = [cf_by[p]['invalid_tool_call_rate']['mean'] - cb_by[p]['invalid_tool_call_rate']['mean'] for p in cal_policies]

cal_mean_success_delta = sum(cal_succ) / len(cal_succ) if cal_succ else 0.0

cal_mean_invalid_delta = sum(cal_inv) / len(cal_inv) if cal_inv else 0.0

cal_desirable = (cal_mean_success_delta >= 0.0) and (cal_mean_invalid_delta <= 0.0)



print('\n=== Comparator Calibration Summary ===')

print('policies:', cal_policies)

print(f'mean success delta (ft-base): {cal_mean_success_delta:+.4f}')

print(f'mean invalid delta (ft-base): {cal_mean_invalid_delta:+.4f}')

print('matrix portfolio verdict:', cm.get('portfolio_verdict'))

print('desirable outcome hit:', cal_desirable)
code
#VSC-fa9df495
python
print('CAL_LABEL_PREFIX =', CAL_LABEL_PREFIX)

print(f'calibration mean success delta (ft-base): {cal_mean_success_delta:+.4f}')

print(f'calibration mean invalid delta (ft-base): {cal_mean_invalid_delta:+.4f}')

print('calibration matrix portfolio verdict:', cm.get('portfolio_verdict'))

print('calibration desirable outcome hit:', cal_desirable)
markdown
#VSC-eccd45b0
markdown
## Anti-Flatline Mini Sweep



Run a compact set of calibrated arms and pick the best concrete outcome instead of trusting a single flat run.

code
#VSC-16277b32
python
sweep_arms = [

    {

        'name': 'balanced_baseline',

        'policies': ['naive_retry', 'exponential_backoff_jitter', 'majority_vote', 'self_consistency'],

        'faults': {'malformed_schema': 0.06, 'timeout': 0.08},

        'max_attempts': 5,

        'time_budget_ms': 2200,

        'seeds': '11,22,33',

    },

    {

        'name': 'low_fault_long_budget',

        'policies': ['naive_retry', 'exponential_backoff_jitter', 'majority_vote', 'self_consistency'],

        'faults': {'malformed_schema': 0.03, 'timeout': 0.04},

        'max_attempts': 6,

        'time_budget_ms': 2800,

        'seeds': '11,22,33',

    },

    {

        'name': 'retry_only_high_attempts',

        'policies': ['naive_retry', 'exponential_backoff_jitter'],

        'faults': {'malformed_schema': 0.04, 'timeout': 0.06},

        'max_attempts': 7,

        'time_budget_ms': 2600,

        'seeds': '11,22,33',

    },

    {

        'name': 'high_fault_stress',

        'policies': ['naive_retry', 'exponential_backoff_jitter', 'majority_vote'],

        'faults': {'malformed_schema': 0.12, 'timeout': 0.12},

        'max_attempts': 5,

        'time_budget_ms': 2400,

        'seeds': '11,22,33',

    },

]



sweep_results = []

for arm in sweep_arms:

    arm_name = arm['name']

    arm_label = f"{LABEL_PREFIX}-sweep-{arm_name}"

    arm_cfg_path = REPO_ROOT / 'configs' / f'toolset_reliability_{arm_name}.json'



    arm_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

    arm_cfg['policies'] = arm['policies']

    arm_faults = dict(arm_cfg.get('fault_probabilities', {}))

    arm_faults['malformed_schema'] = float(arm['faults']['malformed_schema'])

    arm_faults['timeout'] = float(arm['faults']['timeout'])

    arm_cfg['fault_probabilities'] = arm_faults

    arm_cfg['max_attempts'] = int(arm['max_attempts'])

    arm_cfg['time_budget_ms'] = int(arm['time_budget_ms'])

    arm_cfg_path.write_text(json.dumps(arm_cfg, indent=2) + '\n', encoding='utf-8')



    run_cmd = [

        'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

        '--config', str(arm_cfg_path),

        '--seeds', arm['seeds'],

        '--base-planner-config', BASE_PLANNER,

        '--ft-planner-config', FT_PLANNER,

        '--label-prefix', arm_label,

    ]

    print('\n=== Running arm:', arm_name, '===')

    print('Command:', ' '.join(run_cmd))

    p = subprocess.Popen(run_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        print('[arm-fail]', arm_name, 'rc=', rc)

        sweep_results.append({

            'arm': arm_name,

            'label': arm_label,

            'rc': rc,

            'success_delta': None,

            'invalid_delta': None,

            'verdict': 'ERROR',

            'score': -999.0,

        })

        continue



    run_root = REPO_ROOT / 'runs'

    bpath = run_root / f'{arm_label}-base-ms' / 'multi_seed.json'

    fpath = run_root / f'{arm_label}-ft-ms' / 'multi_seed.json'

    mpath = run_root / f'{arm_label}-matrix' / 'matrix.json'

    bobj = json.loads(bpath.read_text(encoding='utf-8'))

    fobj = json.loads(fpath.read_text(encoding='utf-8'))

    mobj = json.loads(mpath.read_text(encoding='utf-8'))



    bby = {r['policy']: r['metrics'] for r in bobj.get('aggregate_policy_metrics', [])}

    fby = {r['policy']: r['metrics'] for r in fobj.get('aggregate_policy_metrics', [])}

    pol = sorted(set(bby) & set(fby))

    succ = [fby[p]['task_success_rate']['mean'] - bby[p]['task_success_rate']['mean'] for p in pol]

    inv = [fby[p]['invalid_tool_call_rate']['mean'] - bby[p]['invalid_tool_call_rate']['mean'] for p in pol]

    mean_s = sum(succ) / len(succ) if succ else 0.0

    mean_i = sum(inv) / len(inv) if inv else 0.0

    verdict = str(mobj.get('portfolio_verdict', 'UNKNOWN'))

    score = mean_s - max(mean_i, 0.0) + (0.02 if verdict == 'PASS' else 0.0)

    sweep_results.append({

        'arm': arm_name,

        'label': arm_label,

        'rc': 0,

        'success_delta': mean_s,

        'invalid_delta': mean_i,

        'verdict': verdict,

        'score': score,

    })



print('\n=== Sweep Result Table ===')

sweep_results = sorted(sweep_results, key=lambda x: x['score'], reverse=True)

for row in sweep_results:

    print(

        row['arm'],

        '| verdict=', row['verdict'],

        '| success_delta=', f"{(row['success_delta'] if row['success_delta'] is not None else float('nan')):+.4f}",

        '| invalid_delta=', f"{(row['invalid_delta'] if row['invalid_delta'] is not None else float('nan')):+.4f}",

        '| score=', f"{row['score']:+.4f}",

    )



best_arm = sweep_results[0] if sweep_results else None

print('\nBEST_ARM =', best_arm)
code
#VSC-dd8d450f
python
print('Sweep rows:', len(sweep_results))

for row in sweep_results:

    print(

        row['arm'],

        '| verdict=', row['verdict'],

        '| success_delta=', f"{(row['success_delta'] if row['success_delta'] is not None else float('nan')):+.4f}",

        '| invalid_delta=', f"{(row['invalid_delta'] if row['invalid_delta'] is not None else float('nan')):+.4f}",

        '| score=', f"{row['score']:+.4f}",

    )

print('BEST_ARM:', best_arm)
markdown
#VSC-83d9763f
markdown
## Enriched Workload Arm (Break Deterministic Flatline)



Use richer task distribution than sample_tasks to increase sensitivity to planner differences.

code
#VSC-526a1af0
python
ENRICHED_LABEL_PREFIX = f"{LABEL_PREFIX}-enriched-cs"

ENRICHED_WORKLOAD = 'workloads/enriched/customer_support.json'

ENRICHED_SEEDS = '11,22,33,44,55'



enriched_cmd = [

    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

    '--config', str(BASE_CONFIG),

    '--workload', ENRICHED_WORKLOAD,

    '--seeds', ENRICHED_SEEDS,

    '--base-planner-config', BASE_PLANNER,

    '--ft-planner-config', FT_PLANNER,

    '--label-prefix', ENRICHED_LABEL_PREFIX,

]

print('Running:', ' '.join(enriched_cmd))

started_enriched = time.time()

enriched_proc = subprocess.Popen(enriched_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

assert enriched_proc.stdout is not None

for line in enriched_proc.stdout:

    print(line, end='')

enriched_rc = enriched_proc.wait()

print(f'[enriched] Total elapsed: {time.time() - started_enriched:.1f}s')

if enriched_rc != 0:

    raise RuntimeError(f'Enriched run failed with code {enriched_rc}')



enriched_root = REPO_ROOT / 'runs'

ebase = enriched_root / f'{ENRICHED_LABEL_PREFIX}-base-ms' / 'multi_seed.json'

eft = enriched_root / f'{ENRICHED_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'

ematrix = enriched_root / f'{ENRICHED_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [ebase, eft, ematrix]:

    print('-', p, 'exists=' + str(p.exists()))



eb = json.loads(ebase.read_text(encoding='utf-8'))

ef = json.loads(eft.read_text(encoding='utf-8'))

em = json.loads(ematrix.read_text(encoding='utf-8'))

eby = {r['policy']: r['metrics'] for r in eb.get('aggregate_policy_metrics', [])}

efy = {r['policy']: r['metrics'] for r in ef.get('aggregate_policy_metrics', [])}

epols = sorted(set(eby) & set(efy))

es = [efy[p]['task_success_rate']['mean'] - eby[p]['task_success_rate']['mean'] for p in epols]

ei = [efy[p]['invalid_tool_call_rate']['mean'] - eby[p]['invalid_tool_call_rate']['mean'] for p in epols]

enriched_mean_success_delta = sum(es) / len(es) if es else 0.0

enriched_mean_invalid_delta = sum(ei) / len(ei) if ei else 0.0

enriched_desirable = (enriched_mean_success_delta > 0.0) and (enriched_mean_invalid_delta <= 0.0)



print('\n=== Enriched Workload Summary ===')

print('workload:', ENRICHED_WORKLOAD)

print('policies:', epols)

print(f'mean success delta (ft-base): {enriched_mean_success_delta:+.4f}')

print(f'mean invalid delta (ft-base): {enriched_mean_invalid_delta:+.4f}')

print('matrix portfolio verdict:', em.get('portfolio_verdict'))

print('strict desirable outcome hit:', enriched_desirable)
code
#VSC-e41ff832
python
print('ENRICHED_LABEL_PREFIX =', ENRICHED_LABEL_PREFIX)

print('workload =', ENRICHED_WORKLOAD)

print(f'enriched mean success delta (ft-base): {enriched_mean_success_delta:+.4f}')

print(f'enriched mean invalid delta (ft-base): {enriched_mean_invalid_delta:+.4f}')

print('enriched matrix portfolio verdict:', em.get('portfolio_verdict'))

print('strict desirable outcome hit:', enriched_desirable)
markdown
#VSC-74a9f3b5
markdown
## Adapter Activity Sanity Check



This check runs base vs base-like (no adapter) on the same enriched workload and compares it against adapter deltas.

code
#VSC-8101279e
python
SANITY_LABEL_PREFIX = f"{LABEL_PREFIX}-sanity-null-adapter"

SANITY_WORKLOAD = ENRICHED_WORKLOAD

SANITY_SEEDS = '11,22,33,44,55'



null_ft_planner_path = REPO_ROOT / 'configs' / 'planners' / 'hf_qwen2_5_3b_ft_nulladapter.json'

null_ft_planner = {

    'type': 'hf_local',

    'name': 'hf_qwen2_5_3b_ft_nulladapter',

    'base_model': 'Qwen/Qwen2.5-3B-Instruct',

}

null_ft_planner_path.write_text(json.dumps(null_ft_planner, indent=2) + '\n', encoding='utf-8')

print('Wrote null-adapter planner:', null_ft_planner_path)



sanity_cmd = [

    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

    '--config', str(BASE_CONFIG),

    '--workload', SANITY_WORKLOAD,

    '--seeds', SANITY_SEEDS,

    '--base-planner-config', BASE_PLANNER,

    '--ft-planner-config', str(null_ft_planner_path.relative_to(REPO_ROOT)),

    '--label-prefix', SANITY_LABEL_PREFIX,

]

print('Running:', ' '.join(sanity_cmd))

started_sanity = time.time()

sanity_proc = subprocess.Popen(sanity_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

assert sanity_proc.stdout is not None

for line in sanity_proc.stdout:

    print(line, end='')

sanity_rc = sanity_proc.wait()

print(f'[sanity] Total elapsed: {time.time() - started_sanity:.1f}s')

if sanity_rc != 0:

    raise RuntimeError(f'Sanity run failed with code {sanity_rc}')



sanity_root = REPO_ROOT / 'runs'

sbase = sanity_root / f'{SANITY_LABEL_PREFIX}-base-ms' / 'multi_seed.json'

sft = sanity_root / f'{SANITY_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'

smatrix = sanity_root / f'{SANITY_LABEL_PREFIX}-matrix' / 'matrix.json'

sb = json.loads(sbase.read_text(encoding='utf-8'))

sf = json.loads(sft.read_text(encoding='utf-8'))

sm = json.loads(smatrix.read_text(encoding='utf-8'))

sby_b = {r['policy']: r['metrics'] for r in sb.get('aggregate_policy_metrics', [])}

sby_f = {r['policy']: r['metrics'] for r in sf.get('aggregate_policy_metrics', [])}

spolicies = sorted(set(sby_b) & set(sby_f))

ss = [sby_f[p]['task_success_rate']['mean'] - sby_b[p]['task_success_rate']['mean'] for p in spolicies]

si = [sby_f[p]['invalid_tool_call_rate']['mean'] - sby_b[p]['invalid_tool_call_rate']['mean'] for p in spolicies]

sanity_mean_success_delta = sum(ss) / len(ss) if ss else 0.0

sanity_mean_invalid_delta = sum(si) / len(si) if si else 0.0



print('\n=== Sanity Null-Adapter Summary ===')

print('workload:', SANITY_WORKLOAD)

print('policies:', spolicies)

print(f'sanity mean success delta (null-ft - base): {sanity_mean_success_delta:+.4f}')

print(f'sanity mean invalid delta (null-ft - base): {sanity_mean_invalid_delta:+.4f}')

print('sanity matrix portfolio verdict:', sm.get('portfolio_verdict'))



adapter_diff_from_sanity_success = enriched_mean_success_delta - sanity_mean_success_delta

adapter_diff_from_sanity_invalid = enriched_mean_invalid_delta - sanity_mean_invalid_delta

adapter_active_signal = (abs(adapter_diff_from_sanity_success) > 0.003) or (abs(adapter_diff_from_sanity_invalid) > 0.003)



print('\n=== Adapter Activity Inference ===')

print(f'adapter run success delta: {enriched_mean_success_delta:+.4f}')

print(f'null-adapter success delta: {sanity_mean_success_delta:+.4f}')

print(f'delta-of-deltas success: {adapter_diff_from_sanity_success:+.4f}')

print(f'adapter run invalid delta: {enriched_mean_invalid_delta:+.4f}')

print(f'null-adapter invalid delta: {sanity_mean_invalid_delta:+.4f}')

print(f'delta-of-deltas invalid: {adapter_diff_from_sanity_invalid:+.4f}')

print('adapter activity signal (>0.003 absolute):', adapter_active_signal)
code
#VSC-11028f00
python
print('SANITY_LABEL_PREFIX =', SANITY_LABEL_PREFIX)

print(f'sanity success delta: {sanity_mean_success_delta:+.4f}')

print(f'sanity invalid delta: {sanity_mean_invalid_delta:+.4f}')

print('sanity matrix verdict:', sm.get('portfolio_verdict'))

print(f'adapter success delta: {enriched_mean_success_delta:+.4f}')

print(f'adapter invalid delta: {enriched_mean_invalid_delta:+.4f}')

print(f'delta-of-deltas success: {adapter_diff_from_sanity_success:+.4f}')

print(f'delta-of-deltas invalid: {adapter_diff_from_sanity_invalid:+.4f}')

print('adapter activity signal (>0.003 absolute):', adapter_active_signal)
markdown
#VSC-67b6511e
markdown
## Multi-Toolset Enriched Evaluator



This evaluator runs enriched workloads per toolset and compares adapter deltas against a null-adapter control.

code
#VSC-39ffc89d
python
ENRICHED_TOOLSETS = ['customer_support', 'ecommerce_ops', 'fintech_risk']

EVAL_SEEDS = '11,22,33'

NULL_FT_PLANNER = REPO_ROOT / 'configs' / 'planners' / 'hf_qwen2_5_3b_ft_nulladapter.json'

if not NULL_FT_PLANNER.exists():

    NULL_FT_PLANNER.write_text(

        json.dumps(

            {

                'type': 'hf_local',

                'name': 'hf_qwen2_5_3b_ft_nulladapter',

                'base_model': 'Qwen/Qwen2.5-3B-Instruct',

            },

            indent=2,

        )

        + '\n',

        encoding='utf-8',

    )



def run_ms(workload_path: str, planner_cfg: str, label: str) -> None:

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(BASE_CONFIG),

        '--workload', workload_path,

        '--seeds', EVAL_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    p = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        raise RuntimeError(f'multi-seed failed for {label} with code {rc}')



def load_policy_means(run_label: str) -> dict:

    path = REPO_ROOT / 'runs' / run_label / 'multi_seed.json'

    obj = json.loads(path.read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}



rows = []

for toolset in ENRICHED_TOOLSETS:

    workload = f'workloads/enriched/{toolset}.json'

    base_label = f'{LABEL_PREFIX}-mts-{toolset}-base'

    ft_label = f'{LABEL_PREFIX}-mts-{toolset}-ft'

    null_label = f'{LABEL_PREFIX}-mts-{toolset}-nullft'



    run_ms(workload, BASE_PLANNER, base_label)

    run_ms(workload, FT_PLANNER, ft_label)

    run_ms(workload, str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), null_label)



    base_metrics = load_policy_means(base_label)

    ft_metrics = load_policy_means(ft_label)

    null_metrics = load_policy_means(null_label)

    pol = sorted(set(base_metrics) & set(ft_metrics) & set(null_metrics))



    ft_success = [(ft_metrics[p]['task_success_rate']['mean'] - base_metrics[p]['task_success_rate']['mean']) for p in pol]

    ft_invalid = [(ft_metrics[p]['invalid_tool_call_rate']['mean'] - base_metrics[p]['invalid_tool_call_rate']['mean']) for p in pol]

    null_success = [(null_metrics[p]['task_success_rate']['mean'] - base_metrics[p]['task_success_rate']['mean']) for p in pol]

    null_invalid = [(null_metrics[p]['invalid_tool_call_rate']['mean'] - base_metrics[p]['invalid_tool_call_rate']['mean']) for p in pol]



    ft_s = sum(ft_success) / len(ft_success) if ft_success else 0.0

    ft_i = sum(ft_invalid) / len(ft_invalid) if ft_invalid else 0.0

    null_s = sum(null_success) / len(null_success) if null_success else 0.0

    null_i = sum(null_invalid) / len(null_invalid) if null_invalid else 0.0



    adapter_adv_s = ft_s - null_s

    adapter_adv_i = ft_i - null_i



    rows.append(

        {

            'toolset': toolset,

            'ft_success_delta': ft_s,

            'ft_invalid_delta': ft_i,

            'null_success_delta': null_s,

            'null_invalid_delta': null_i,

            'adapter_adv_success': adapter_adv_s,

            'adapter_adv_invalid': adapter_adv_i,

            'strict_pass': (ft_s >= 0.01 and ft_i <= 0.0),

            'control_beating': (adapter_adv_s > 0.003 and adapter_adv_i <= 0.0),

        }

    )



print('\n=== Multi-Toolset Enriched Results ===')

for r in rows:

    print(

        r['toolset'],

        '| ft_s=', f"{r['ft_success_delta']:+.4f}",

        '| ft_i=', f"{r['ft_invalid_delta']:+.4f}",

        '| null_s=', f"{r['null_success_delta']:+.4f}",

        '| null_i=', f"{r['null_invalid_delta']:+.4f}",

        '| adv_s=', f"{r['adapter_adv_success']:+.4f}",

        '| adv_i=', f"{r['adapter_adv_invalid']:+.4f}",

        '| strict_pass=', r['strict_pass'],

        '| control_beating=', r['control_beating'],

    )



strict_pass_count = sum(1 for r in rows if r['strict_pass'])

control_beating_count = sum(1 for r in rows if r['control_beating'])

portfolio_positive = strict_pass_count >= 2

print('\nstrict_pass_count =', strict_pass_count)

print('control_beating_count =', control_beating_count)

print('portfolio_positive (>=2 strict passes) =', portfolio_positive)

mts_summary = {

    'rows': rows,

    'strict_pass_count': strict_pass_count,

    'control_beating_count': control_beating_count,

    'portfolio_positive': portfolio_positive,

}

mts_path = REPO_ROOT / 'runs' / f'{LABEL_PREFIX}-mts-summary.json'

mts_path.write_text(json.dumps(mts_summary, indent=2) + '\n', encoding='utf-8')

print('Saved summary:', mts_path)
code
#VSC-0e871823
python
print('Multi-toolset rows:', len(mts_summary['rows']))

for r in mts_summary['rows']:

    print(

        r['toolset'],

        '| ft_s=', f"{r['ft_success_delta']:+.4f}",

        '| ft_i=', f"{r['ft_invalid_delta']:+.4f}",

        '| null_s=', f"{r['null_success_delta']:+.4f}",

        '| null_i=', f"{r['null_invalid_delta']:+.4f}",

        '| adv_s=', f"{r['adapter_adv_success']:+.4f}",

        '| adv_i=', f"{r['adapter_adv_invalid']:+.4f}",

        '| strict_pass=', r['strict_pass'],

        '| control_beating=', r['control_beating'],

    )

print('strict_pass_count =', mts_summary['strict_pass_count'])

print('control_beating_count =', mts_summary['control_beating_count'])

print('portfolio_positive =', mts_summary['portfolio_positive'])

print('summary_path =', REPO_ROOT / 'runs' / f'{LABEL_PREFIX}-mts-summary.json')
markdown
#VSC-43fc632f
markdown
## Failure-Focused Slice Miner and Re-Eval



Mine task IDs where adapter underperforms base on a single run, build a hard-case slice workload, then re-evaluate base vs ft vs null-adapter on that slice.

code
#VSC-4ddadea0
python
HARD_TOOLSET = 'customer_support'

HARD_WORKLOAD_PATH = REPO_ROOT / 'workloads' / 'enriched' / f'{HARD_TOOLSET}.json'

HARD_SEED = '41'

HARD_SLICE_SEEDS = '11,22,33,44,55'



hard_base_label = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-single-base"

hard_ft_label = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-single-ft"



def run_single(workload_path: str, planner_cfg: str, label: str) -> dict:

    cmd = [

        'uv', 'run', 'tcrb', 'run',

        '--config', str(BASE_CONFIG),

        '--workload', workload_path,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    env = os.environ.copy()

    env['PYTHONHASHSEED'] = HARD_SEED

    print('Running:', ' '.join(cmd))

    p = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        raise RuntimeError(f'single run failed for {label} with code {rc}')

    payload_path = REPO_ROOT / 'runs' / label / 'result.json'

    return json.loads(payload_path.read_text(encoding='utf-8'))



base_payload = run_single(str(HARD_WORKLOAD_PATH.relative_to(REPO_ROOT)), BASE_PLANNER, hard_base_label)

ft_payload = run_single(str(HARD_WORKLOAD_PATH.relative_to(REPO_ROOT)), FT_PLANNER, hard_ft_label)



base_task = {r['task_id']: r for r in base_payload.get('task_results', [])}

ft_task = {r['task_id']: r for r in ft_payload.get('task_results', [])}

common_ids = sorted(set(base_task) & set(ft_task))



hard_ids = []

for tid in common_ids:

    b = base_task[tid]

    f = ft_task[tid]

    base_succ = bool(b.get('success'))

    ft_succ = bool(f.get('success'))

    base_inv = any(bool(a.get('invalid_tool_call')) for a in b.get('attempts', []))

    ft_inv = any(bool(a.get('invalid_tool_call')) for a in f.get('attempts', []))

    if (base_succ and not ft_succ) or ((not base_inv) and ft_inv):

        hard_ids.append(tid)



if not hard_ids:

    fallback = []

    for tid in common_ids:

        b = base_task[tid]

        f = ft_task[tid]

        penalty = 0

        if bool(b.get('success')) and not bool(f.get('success')):

            penalty += 3

        if any(bool(a.get('invalid_tool_call')) for a in f.get('attempts', [])):

            penalty += 2

        if float(f.get('total_latency_ms', 0.0)) > float(b.get('total_latency_ms', 0.0)):

            penalty += 1

        fallback.append((penalty, tid))

    fallback.sort(reverse=True)

    hard_ids = [tid for _, tid in fallback[: max(6, min(12, len(fallback)))]]



print('Selected hard task ids:', hard_ids)



hard_workload = json.loads(HARD_WORKLOAD_PATH.read_text(encoding='utf-8'))

task_lookup = {t['task_id']: t for t in hard_workload.get('tasks', [])}

slice_tasks = [task_lookup[tid] for tid in hard_ids if tid in task_lookup]

if not slice_tasks:

    raise RuntimeError('No tasks selected for hard slice workload.')



hard_slice_payload = {

    'toolset_id': f"{hard_workload.get('toolset_id', HARD_TOOLSET)}_hard_slice",

    'tools': hard_workload.get('tools', []),

    'tasks': slice_tasks,

}

hard_slice_path = REPO_ROOT / 'workloads' / 'enriched' / f'{HARD_TOOLSET}_hard_slice.json'

hard_slice_path.write_text(json.dumps(hard_slice_payload, indent=2) + '\n', encoding='utf-8')

print('Wrote hard-slice workload:', hard_slice_path)

print('Hard-slice task count:', len(slice_tasks))



hard_eval_base = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-base-ms"

hard_eval_ft = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-ft-ms"

hard_eval_null = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-null-ms"



def run_ms(workload_path: str, planner_cfg: str, label: str) -> None:

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(BASE_CONFIG),

        '--workload', workload_path,

        '--seeds', HARD_SLICE_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    p = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        raise RuntimeError(f'multi-seed failed for {label} with code {rc}')



run_ms(str(hard_slice_path.relative_to(REPO_ROOT)), BASE_PLANNER, hard_eval_base)

run_ms(str(hard_slice_path.relative_to(REPO_ROOT)), FT_PLANNER, hard_eval_ft)

run_ms(str(hard_slice_path.relative_to(REPO_ROOT)), str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), hard_eval_null)



def read_ms(label: str) -> dict:

    p = REPO_ROOT / 'runs' / label / 'multi_seed.json'

    obj = json.loads(p.read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}



hm_base = read_ms(hard_eval_base)

hm_ft = read_ms(hard_eval_ft)

hm_null = read_ms(hard_eval_null)

hpol = sorted(set(hm_base) & set(hm_ft) & set(hm_null))



h_ft_s = [hm_ft[p]['task_success_rate']['mean'] - hm_base[p]['task_success_rate']['mean'] for p in hpol]

h_ft_i = [hm_ft[p]['invalid_tool_call_rate']['mean'] - hm_base[p]['invalid_tool_call_rate']['mean'] for p in hpol]

h_n_s = [hm_null[p]['task_success_rate']['mean'] - hm_base[p]['task_success_rate']['mean'] for p in hpol]

h_n_i = [hm_null[p]['invalid_tool_call_rate']['mean'] - hm_base[p]['invalid_tool_call_rate']['mean'] for p in hpol]



hard_ft_success_delta = sum(h_ft_s) / len(h_ft_s) if h_ft_s else 0.0

hard_ft_invalid_delta = sum(h_ft_i) / len(h_ft_i) if h_ft_i else 0.0

hard_null_success_delta = sum(h_n_s) / len(h_n_s) if h_n_s else 0.0

hard_null_invalid_delta = sum(h_n_i) / len(h_n_i) if h_n_i else 0.0

hard_adv_success = hard_ft_success_delta - hard_null_success_delta

hard_adv_invalid = hard_ft_invalid_delta - hard_null_invalid_delta

hard_strict_pass = hard_ft_success_delta >= 0.01 and hard_ft_invalid_delta <= 0.0

hard_control_beating = hard_adv_success > 0.003 and hard_adv_invalid <= 0.0



print('\n=== Hard-Slice Re-Eval Summary ===')

print('toolset:', HARD_TOOLSET)

print('task_count:', len(slice_tasks))

print('policies:', hpol)

print(f'ft success delta: {hard_ft_success_delta:+.4f}')

print(f'ft invalid delta: {hard_ft_invalid_delta:+.4f}')

print(f'null success delta: {hard_null_success_delta:+.4f}')

print(f'null invalid delta: {hard_null_invalid_delta:+.4f}')

print(f'adapter advantage success: {hard_adv_success:+.4f}')

print(f'adapter advantage invalid: {hard_adv_invalid:+.4f}')

print('strict_pass:', hard_strict_pass)

print('control_beating:', hard_control_beating)
code
#VSC-c75f0659
python
print('hard-slice workload path =', hard_slice_path)

print('hard task ids =', hard_ids)

print('hard task count =', len(slice_tasks))

print(f'hard ft success delta: {hard_ft_success_delta:+.4f}')

print(f'hard ft invalid delta: {hard_ft_invalid_delta:+.4f}')

print(f'hard null success delta: {hard_null_success_delta:+.4f}')

print(f'hard null invalid delta: {hard_null_invalid_delta:+.4f}')

print(f'hard adapter advantage success: {hard_adv_success:+.4f}')

print(f'hard adapter advantage invalid: {hard_adv_invalid:+.4f}')

print('hard strict_pass =', hard_strict_pass)

print('hard control_beating =', hard_control_beating)
markdown
#VSC-af648131
markdown
## Adapter Integrity Audit



Validate that adapter files exist, have non-trivial tensor norms, and are wired to the finetuned planner config.

code
#VSC-498fb623
python
from pathlib import Path

import json



adapter_dir = REPO_ROOT / 'outputs' / 'ft-notebook' / 'final'

adapter_cfg_path = adapter_dir / 'adapter_config.json'

adapter_weights_path = adapter_dir / 'adapter_model.safetensors'

ft_planner_cfg_path = REPO_ROOT / FT_PLANNER



print('adapter_dir =', adapter_dir)

print('adapter_config exists =', adapter_cfg_path.exists())

print('adapter_weights exists =', adapter_weights_path.exists())

print('adapter_weights size bytes =', adapter_weights_path.stat().st_size if adapter_weights_path.exists() else -1)



planner_adapter_path = None

if ft_planner_cfg_path.exists():

    planner_obj = json.loads(ft_planner_cfg_path.read_text(encoding='utf-8'))

    planner_adapter_path = str(planner_obj.get('adapter_path', '')).strip()

    print('ft planner config path =', ft_planner_cfg_path)

    print('ft planner adapter_path =', planner_adapter_path)

    print('ft planner adapter target exists =', (REPO_ROOT / planner_adapter_path).exists() if planner_adapter_path else False)

else:

    print('ft planner config missing:', ft_planner_cfg_path)



adapter_cfg = {}

if adapter_cfg_path.exists():

    adapter_cfg = json.loads(adapter_cfg_path.read_text(encoding='utf-8'))

    print('adapter base_model_name_or_path =', adapter_cfg.get('base_model_name_or_path'))

    print('adapter r =', adapter_cfg.get('r'))

    print('adapter lora_alpha =', adapter_cfg.get('lora_alpha'))

    print('adapter target_modules =', adapter_cfg.get('target_modules'))



tensor_count = 0

lora_tensor_count = 0

nonzero_tensor_count = 0

sample_keys = []

sample_stats = []



if adapter_weights_path.exists():

    try:

        from safetensors import safe_open

        import torch



        with safe_open(str(adapter_weights_path), framework='pt', device='cpu') as f:

            keys = list(f.keys())

            tensor_count = len(keys)

            for k in keys:

                if 'lora_' in k:

                    lora_tensor_count += 1

                t = f.get_tensor(k)

                max_abs = float(t.abs().max().item()) if t.numel() > 0 else 0.0

                if max_abs > 1e-9:

                    nonzero_tensor_count += 1

                if len(sample_keys) < 8:

                    sample_keys.append(k)

                    mean_abs = float(t.abs().mean().item()) if t.numel() > 0 else 0.0

                    sample_stats.append({'key': k, 'shape': list(t.shape), 'mean_abs': mean_abs, 'max_abs': max_abs})

        print('tensor_count =', tensor_count)

        print('lora_tensor_count =', lora_tensor_count)

        print('nonzero_tensor_count =', nonzero_tensor_count)

        print('nonzero_ratio =', (nonzero_tensor_count / tensor_count) if tensor_count else 0.0)

        print('sample_stats =')

        for row in sample_stats:

            print(' -', row)

    except Exception as exc:

        print('safetensor inspection error:', exc)



adapter_materialized = (

    adapter_cfg_path.exists()

    and adapter_weights_path.exists()

    and tensor_count > 0

    and nonzero_tensor_count > 0

)

adapter_wired = bool(planner_adapter_path) and (REPO_ROOT / planner_adapter_path).exists()



print('adapter_materialized =', adapter_materialized)

print('adapter_wired =', adapter_wired)

print('adapter_integrity_pass =', adapter_materialized and adapter_wired)
markdown
#VSC-d5dc5152
markdown
## Model-Sensitive Policy Re-Eval



Re-run enriched multi-toolset evaluation with model-sensitive policies only to avoid heuristic-policy masking.

code
#VSC-82aeee5c
python
NULL_FT_PLANNER = REPO_ROOT / 'configs' / 'planners' / 'hf_qwen2_5_3b_ft_nulladapter.json'


if not NULL_FT_PLANNER.exists():


    NULL_FT_PLANNER.write_text(


        json.dumps(


            {


                'type': 'hf_local',


                'name': 'hf_qwen2_5_3b_ft_nulladapter',


                'base_model': 'Qwen/Qwen2.5-3B-Instruct',


            },


            indent=2,


        )


        + '\n',


        encoding='utf-8',


    )




print('NULL_FT_PLANNER =', NULL_FT_PLANNER)
code
#VSC-5afc2418
python
MS_ONLY_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_model_sensitive_only.json'

base_cfg_obj = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

base_cfg_obj['policies'] = ['naive_retry', 'exponential_backoff_jitter']

faults = dict(base_cfg_obj.get('fault_probabilities', {}))

faults['malformed_schema'] = max(float(faults.get('malformed_schema', 0.06)), 0.08)

faults['timeout'] = max(float(faults.get('timeout', 0.08)), 0.10)

base_cfg_obj['fault_probabilities'] = faults

base_cfg_obj['max_attempts'] = max(int(base_cfg_obj.get('max_attempts', 4)), 5)

base_cfg_obj['time_budget_ms'] = max(int(base_cfg_obj.get('time_budget_ms', 1800)), 2200)

MS_ONLY_CONFIG.write_text(json.dumps(base_cfg_obj, indent=2) + '\n', encoding='utf-8')

print('Wrote config:', MS_ONLY_CONFIG)



MS_TOOLSETS = ['customer_support', 'ecommerce_ops', 'fintech_risk']

MS_SEEDS = '11,22,33'

ms_rows = []



def run_ms_only(workload_path: str, planner_cfg: str, label: str):

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(MS_ONLY_CONFIG),

        '--workload', workload_path,

        '--seeds', MS_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    p = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        raise RuntimeError(f'multi-seed failed for {label} with code {rc}')



def read_metrics(label: str):

    obj = json.loads((REPO_ROOT / 'runs' / label / 'multi_seed.json').read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}



for toolset in MS_TOOLSETS:

    wl = f'workloads/enriched/{toolset}.json'

    b_label = f"{LABEL_PREFIX}-msonly-{toolset}-base"

    f_label = f"{LABEL_PREFIX}-msonly-{toolset}-ft"

    n_label = f"{LABEL_PREFIX}-msonly-{toolset}-null"



    run_ms_only(wl, BASE_PLANNER, b_label)

    run_ms_only(wl, FT_PLANNER, f_label)

    run_ms_only(wl, str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), n_label)



    bm = read_metrics(b_label)

    fm = read_metrics(f_label)

    nm = read_metrics(n_label)

    policies = sorted(set(bm) & set(fm) & set(nm))

    fs = [fm[p]['task_success_rate']['mean'] - bm[p]['task_success_rate']['mean'] for p in policies]

    fi = [fm[p]['invalid_tool_call_rate']['mean'] - bm[p]['invalid_tool_call_rate']['mean'] for p in policies]

    ns = [nm[p]['task_success_rate']['mean'] - bm[p]['task_success_rate']['mean'] for p in policies]

    ni = [nm[p]['invalid_tool_call_rate']['mean'] - bm[p]['invalid_tool_call_rate']['mean'] for p in policies]

    ft_s = sum(fs) / len(fs) if fs else 0.0

    ft_i = sum(fi) / len(fi) if fi else 0.0

    n_s = sum(ns) / len(ns) if ns else 0.0

    n_i = sum(ni) / len(ni) if ni else 0.0

    adv_s = ft_s - n_s

    adv_i = ft_i - n_i

    ms_rows.append({

        'toolset': toolset,

        'ft_success_delta': ft_s,

        'ft_invalid_delta': ft_i,

        'null_success_delta': n_s,

        'null_invalid_delta': n_i,

        'adapter_adv_success': adv_s,

        'adapter_adv_invalid': adv_i,

        'strict_pass': (ft_s >= 0.01 and ft_i <= 0.0),

        'control_beating': (adv_s > 0.003 and adv_i <= 0.0),

    })



print('\n=== Model-Sensitive Re-Eval ===')

for r in ms_rows:

    print(

        r['toolset'],

        '| ft_s=', f"{r['ft_success_delta']:+.4f}",

        '| ft_i=', f"{r['ft_invalid_delta']:+.4f}",

        '| null_s=', f"{r['null_success_delta']:+.4f}",

        '| null_i=', f"{r['null_invalid_delta']:+.4f}",

        '| adv_s=', f"{r['adapter_adv_success']:+.4f}",

        '| adv_i=', f"{r['adapter_adv_invalid']:+.4f}",

        '| strict_pass=', r['strict_pass'],

        '| control_beating=', r['control_beating'],

    )



ms_strict_pass_count = sum(1 for r in ms_rows if r['strict_pass'])

ms_control_beating_count = sum(1 for r in ms_rows if r['control_beating'])

ms_portfolio_positive = ms_strict_pass_count >= 2

print('ms_strict_pass_count =', ms_strict_pass_count)

print('ms_control_beating_count =', ms_control_beating_count)

print('ms_portfolio_positive =', ms_portfolio_positive)
code
#VSC-b63eec10
python
for r in ms_rows:

    print(

        r['toolset'],

        '| ft_s=', f"{r['ft_success_delta']:+.4f}",

        '| ft_i=', f"{r['ft_invalid_delta']:+.4f}",

        '| null_s=', f"{r['null_success_delta']:+.4f}",

        '| null_i=', f"{r['null_invalid_delta']:+.4f}",

        '| adv_s=', f"{r['adapter_adv_success']:+.4f}",

        '| adv_i=', f"{r['adapter_adv_invalid']:+.4f}",

        '| strict_pass=', r['strict_pass'],

        '| control_beating=', r['control_beating'],

    )

print('ms_strict_pass_count =', ms_strict_pass_count)

print('ms_control_beating_count =', ms_control_beating_count)

print('ms_portfolio_positive =', ms_portfolio_positive)
markdown
#VSC-8392fe72
markdown
## Synthetic Ambiguous Toolset + Eval



Create a harder ambiguous-routing toolset where candidate tools share overlapping semantics and evaluate base vs ft vs null-adapter.

code
#VSC-ff459b51
python
SYN_TOOLSET_ID = 'ambiguous_ops'

SYN_WORKLOAD_PATH = REPO_ROOT / 'workloads' / 'enriched' / 'ambiguous_ops.json'

SYN_CONFIG_PATH = REPO_ROOT / 'configs' / 'toolset_reliability_ambiguous_ops.json'

SYN_SEEDS = '11,22,33,44,55'



synthetic_tools = [

    {

        'name': 'account_snapshot_api',

        'description': 'Returns account status, balance, and flags.',

        'base_latency_ms': 250,

        'jitter_ms': 80,

        'timeout_ms': 900,

        'schema_fields': ['account_id', 'status', 'balance', 'flags'],

        'fault_multipliers': {'timeout': 1.1},

    },

    {

        'name': 'account_risk_profile_api',

        'description': 'Returns account risk profile and score.',

        'base_latency_ms': 280,

        'jitter_ms': 85,

        'timeout_ms': 950,

        'schema_fields': ['account_id', 'risk_score', 'risk_level', 'flags'],

        'fault_multipliers': {'malformed_schema': 1.2},

    },

    {

        'name': 'payment_status_api',

        'description': 'Returns payment state and settlement info.',

        'base_latency_ms': 260,

        'jitter_ms': 80,

        'timeout_ms': 900,

        'schema_fields': ['payment_id', 'status', 'settlement_eta', 'amount'],

        'fault_multipliers': {'network_failure': 1.1},

    },

    {

        'name': 'payment_dispute_api',

        'description': 'Returns dispute eligibility and reason codes.',

        'base_latency_ms': 300,

        'jitter_ms': 90,

        'timeout_ms': 1000,

        'schema_fields': ['payment_id', 'eligible', 'reason_code', 'policy_version'],

        'fault_multipliers': {'contract_drift': 1.2},

    },

    {

        'name': 'identity_check_api',

        'description': 'Returns verification state and method confidence.',

        'base_latency_ms': 320,

        'jitter_ms': 95,

        'timeout_ms': 1100,

        'schema_fields': ['user_id', 'verified', 'method', 'confidence'],

        'fault_multipliers': {'timeout': 1.2},

    },

    {

        'name': 'session_forensics_api',

        'description': 'Returns suspicious session events and anomalies.',

        'base_latency_ms': 340,

        'jitter_ms': 100,

        'timeout_ms': 1150,

        'schema_fields': ['user_id', 'events', 'anomalies', 'risk_level'],

        'fault_multipliers': {'malformed_schema': 1.1},

    },

    {

        'name': 'knowledge_semantic_api',

        'description': 'Returns semantic answer with citations and confidence.',

        'base_latency_ms': 390,

        'jitter_ms': 120,

        'timeout_ms': 1250,

        'schema_fields': ['answer', 'sources', 'confidence', 'policy_version'],

        'fault_multipliers': {'timeout': 1.2},

    },

    {

        'name': 'knowledge_keyword_api',

        'description': 'Returns keyword match answer with citations.',

        'base_latency_ms': 310,

        'jitter_ms': 95,

        'timeout_ms': 1000,

        'schema_fields': ['answer', 'sources', 'confidence', 'query'],

        'fault_multipliers': {'network_failure': 1.1},

    },

]



synthetic_tasks = [

    {

        'task_id': 'ao-001',

        'user_query': 'Need live account snapshot before changing limits.',

        'primary_tool': 'account_snapshot_api',

        'fallback_tools': ['account_risk_profile_api', 'session_forensics_api'],

        'required_schema': ['account_id', 'status', 'balance', 'flags'],

    },

    {

        'task_id': 'ao-002',

        'user_query': 'Need account risk profile and level for compliance hold.',

        'primary_tool': 'account_risk_profile_api',

        'fallback_tools': ['account_snapshot_api', 'session_forensics_api'],

        'required_schema': ['account_id', 'risk_score', 'risk_level', 'flags'],

    },

    {

        'task_id': 'ao-003',

        'user_query': 'Is payment p-1149 settled and what is ETA?',

        'primary_tool': 'payment_status_api',

        'fallback_tools': ['payment_dispute_api', 'knowledge_keyword_api'],

        'required_schema': ['payment_id', 'status', 'settlement_eta', 'amount'],

    },

    {

        'task_id': 'ao-004',

        'user_query': 'Can payment p-772 be disputed and under which policy?',

        'primary_tool': 'payment_dispute_api',

        'fallback_tools': ['payment_status_api', 'knowledge_semantic_api'],

        'required_schema': ['payment_id', 'eligible', 'reason_code', 'policy_version'],

    },

    {

        'task_id': 'ao-005',

        'user_query': 'Verify user identity for risky payout release.',

        'primary_tool': 'identity_check_api',

        'fallback_tools': ['session_forensics_api', 'account_risk_profile_api'],

        'required_schema': ['user_id', 'verified', 'method', 'confidence'],

    },

    {

        'task_id': 'ao-006',

        'user_query': 'Investigate suspicious session trail before unlock.',

        'primary_tool': 'session_forensics_api',

        'fallback_tools': ['identity_check_api', 'account_snapshot_api'],

        'required_schema': ['user_id', 'events', 'anomalies', 'risk_level'],

    },

    {

        'task_id': 'ao-007',

        'user_query': 'Find best policy answer for delayed settlement exceptions.',

        'primary_tool': 'knowledge_semantic_api',

        'fallback_tools': ['knowledge_keyword_api', 'payment_status_api'],

        'required_schema': ['answer', 'sources', 'confidence', 'policy_version'],

    },

    {

        'task_id': 'ao-008',

        'user_query': 'Keyword lookup for exact dispute reason code table.',

        'primary_tool': 'knowledge_keyword_api',

        'fallback_tools': ['knowledge_semantic_api', 'payment_dispute_api'],

        'required_schema': ['answer', 'sources', 'confidence', 'query'],

    },

]



# Duplicate with paraphrases to increase evaluation surface while preserving schema intent.

for i in range(9, 25):

    src = synthetic_tasks[(i - 1) % 8]

    synthetic_tasks.append(

        {

            'task_id': f"ao-{i:03d}",

            'user_query': src['user_query'] + f" [variant {i}]",

            'primary_tool': src['primary_tool'],

            'fallback_tools': src['fallback_tools'],

            'required_schema': src['required_schema'],

        }

    )



synthetic_workload = {

    'toolset_id': SYN_TOOLSET_ID,

    'tools': synthetic_tools,

    'tasks': synthetic_tasks,

}

SYN_WORKLOAD_PATH.write_text(json.dumps(synthetic_workload, indent=2) + '\n', encoding='utf-8')

print('Wrote synthetic workload:', SYN_WORKLOAD_PATH)

print('Synthetic task count:', len(synthetic_tasks))



syn_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

syn_cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']

syn_faults = dict(syn_cfg.get('fault_probabilities', {}))

syn_faults['malformed_schema'] = max(float(syn_faults.get('malformed_schema', 0.06)), 0.10)

syn_faults['timeout'] = max(float(syn_faults.get('timeout', 0.08)), 0.10)

syn_cfg['fault_probabilities'] = syn_faults

syn_cfg['max_attempts'] = max(int(syn_cfg.get('max_attempts', 4)), 5)

syn_cfg['time_budget_ms'] = max(int(syn_cfg.get('time_budget_ms', 1800)), 2300)

SYN_CONFIG_PATH.write_text(json.dumps(syn_cfg, indent=2) + '\n', encoding='utf-8')

print('Wrote synthetic eval config:', SYN_CONFIG_PATH)



SYN_BASE_LABEL = f"{LABEL_PREFIX}-syn-ambiguous-base"

SYN_FT_LABEL = f"{LABEL_PREFIX}-syn-ambiguous-ft"

SYN_NULL_LABEL = f"{LABEL_PREFIX}-syn-ambiguous-null"



def run_syn(planner_cfg: str, label: str):

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(SYN_CONFIG_PATH),

        '--workload', str(SYN_WORKLOAD_PATH.relative_to(REPO_ROOT)),

        '--seeds', SYN_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    completed = subprocess.run(cmd, text=True, capture_output=True, cwd=str(REPO_ROOT), check=False)

    if completed.returncode != 0:

        print((completed.stdout or '')[-2000:])

        print((completed.stderr or '')[-2000:])

        raise RuntimeError(f'synthetic run failed for {label} with code {completed.returncode}')

    print('[ok]', label)



run_syn(BASE_PLANNER, SYN_BASE_LABEL)

run_syn(FT_PLANNER, SYN_FT_LABEL)

run_syn(str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), SYN_NULL_LABEL)



def load_ms(label: str):

    payload = json.loads((REPO_ROOT / 'runs' / label / 'multi_seed.json').read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in payload.get('aggregate_policy_metrics', [])}



sb = load_ms(SYN_BASE_LABEL)

sf = load_ms(SYN_FT_LABEL)

sn = load_ms(SYN_NULL_LABEL)

sp = sorted(set(sb) & set(sf) & set(sn))

s_ft_s = [sf[p]['task_success_rate']['mean'] - sb[p]['task_success_rate']['mean'] for p in sp]

s_ft_i = [sf[p]['invalid_tool_call_rate']['mean'] - sb[p]['invalid_tool_call_rate']['mean'] for p in sp]

s_n_s = [sn[p]['task_success_rate']['mean'] - sb[p]['task_success_rate']['mean'] for p in sp]

s_n_i = [sn[p]['invalid_tool_call_rate']['mean'] - sb[p]['invalid_tool_call_rate']['mean'] for p in sp]



syn_ft_success_delta = sum(s_ft_s) / len(s_ft_s) if s_ft_s else 0.0

syn_ft_invalid_delta = sum(s_ft_i) / len(s_ft_i) if s_ft_i else 0.0

syn_null_success_delta = sum(s_n_s) / len(s_n_s) if s_n_s else 0.0

syn_null_invalid_delta = sum(s_n_i) / len(s_n_i) if s_n_i else 0.0

syn_adapter_adv_success = syn_ft_success_delta - syn_null_success_delta

syn_adapter_adv_invalid = syn_ft_invalid_delta - syn_null_invalid_delta

syn_strict_pass = syn_ft_success_delta >= 0.01 and syn_ft_invalid_delta <= 0.0

syn_control_beating = syn_adapter_adv_success > 0.003 and syn_adapter_adv_invalid <= 0.0



print('\n=== Synthetic Toolset Eval Summary ===')

print('toolset_id =', SYN_TOOLSET_ID)

print('policies =', sp)

print(f'ft success delta: {syn_ft_success_delta:+.4f}')

print(f'ft invalid delta: {syn_ft_invalid_delta:+.4f}')

print(f'null success delta: {syn_null_success_delta:+.4f}')

print(f'null invalid delta: {syn_null_invalid_delta:+.4f}')

print(f'adapter advantage success: {syn_adapter_adv_success:+.4f}')

print(f'adapter advantage invalid: {syn_adapter_adv_invalid:+.4f}')

print('strict_pass =', syn_strict_pass)

print('control_beating =', syn_control_beating)
markdown
#VSC-e8f0a355
markdown
## Synthetic Label-Leakage Probe Toolset + Eval



This probe uses semantically similar tools and checks whether outcomes are invariant across base/ft/null, indicating task-field leakage dominates model behavior.

code
#VSC-7c60abf9
python
# Ensure Kaggle runtime has latest planner patch from repo before leak-probe run.

sync_cmd = ['git', 'pull', '--ff-only', 'origin', 'main']

print('Running:', ' '.join(sync_cmd))

sync_res = subprocess.run(sync_cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)

if sync_res.stdout:

    print(sync_res.stdout)

if sync_res.returncode != 0:

    if sync_res.stderr:

        print(sync_res.stderr)

    raise RuntimeError(f'git pull failed with code {sync_res.returncode}')



verify_cmd = ['python', '-c', "from pathlib import Path; p=Path('src/tcrb/hf_planner.py'); t=p.read_text(encoding='utf-8'); print('prompt_patch_present=', 'Do not infer labels from task IDs or hidden metadata' in t)"]

verify_res = subprocess.run(verify_cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)

if verify_res.stdout:

    print(verify_res.stdout)

if verify_res.returncode != 0:

    if verify_res.stderr:

        print(verify_res.stderr)

    raise RuntimeError(f'planner verification failed with code {verify_res.returncode}')

code
#VSC-4ce76f25
python
LEAK_TOOLSET_ID = 'ambiguous_ops_leak_probe'

LEAK_WORKLOAD_PATH = REPO_ROOT / 'workloads' / 'enriched' / 'ambiguous_ops_leak_probe.json'

LEAK_CONFIG_PATH = REPO_ROOT / 'configs' / 'toolset_reliability_leak_probe.json'

LEAK_SEEDS = '11,22,33,44,55'



leak_tools = [

    {

        'name': 'resolver_alpha',

        'description': 'General resolver with broad retrieval capability.',

        'base_latency_ms': 280,

        'jitter_ms': 85,

        'timeout_ms': 1000,

        'schema_fields': ['answer', 'sources', 'confidence'],

        'fault_multipliers': {'timeout': 1.0},

    },

    {

        'name': 'resolver_beta',

        'description': 'General resolver with broad retrieval capability.',

        'base_latency_ms': 282,

        'jitter_ms': 85,

        'timeout_ms': 1000,

        'schema_fields': ['answer', 'sources', 'confidence'],

        'fault_multipliers': {'timeout': 1.0},

    },

    {

        'name': 'resolver_gamma',

        'description': 'General resolver with broad retrieval capability.',

        'base_latency_ms': 284,

        'jitter_ms': 85,

        'timeout_ms': 1000,

        'schema_fields': ['answer', 'sources', 'confidence'],

        'fault_multipliers': {'timeout': 1.0},

    },

]



leak_tasks = []

for i in range(1, 31):

    if i % 3 == 1:

        primary = 'resolver_alpha'

        fallbacks = ['resolver_beta', 'resolver_gamma']

    elif i % 3 == 2:

        primary = 'resolver_beta'

        fallbacks = ['resolver_gamma', 'resolver_alpha']

    else:

        primary = 'resolver_gamma'

        fallbacks = ['resolver_alpha', 'resolver_beta']

    leak_tasks.append(

        {

            'task_id': f'lp-{i:03d}',

            'user_query': f'Need best resolution for ambiguous request variant {i}',

            'primary_tool': primary,

            'fallback_tools': fallbacks,

            'required_schema': ['answer', 'sources', 'confidence'],

        }

    )



leak_workload = {'toolset_id': LEAK_TOOLSET_ID, 'tools': leak_tools, 'tasks': leak_tasks}

LEAK_WORKLOAD_PATH.write_text(json.dumps(leak_workload, indent=2) + '\n', encoding='utf-8')

print('Wrote leak-probe workload:', LEAK_WORKLOAD_PATH)



leak_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

leak_cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']

leak_cfg['fault_probabilities'] = {

    'timeout': 0.03,

    'rate_limit': 0.02,

    'malformed_schema': 0.02,

    'contract_drift': 0.02,

    'network_failure': 0.02,

}

leak_cfg['max_attempts'] = 4

leak_cfg['time_budget_ms'] = 1800

LEAK_CONFIG_PATH.write_text(json.dumps(leak_cfg, indent=2) + '\n', encoding='utf-8')

print('Wrote leak-probe config:', LEAK_CONFIG_PATH)



LEAK_BASE_LABEL = f'{LABEL_PREFIX}-leakprobe-base'

LEAK_FT_LABEL = f'{LABEL_PREFIX}-leakprobe-ft'

LEAK_NULL_LABEL = f'{LABEL_PREFIX}-leakprobe-null'



def run_leak(planner_cfg: str, label: str):

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(LEAK_CONFIG_PATH),

        '--workload', str(LEAK_WORKLOAD_PATH.relative_to(REPO_ROOT)),

        '--seeds', LEAK_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    out = subprocess.run(cmd, text=True, capture_output=True, cwd=str(REPO_ROOT), check=False)

    if out.returncode != 0:

        print((out.stdout or '')[-2000:])

        print((out.stderr or '')[-2000:])

        raise RuntimeError(f'Leak probe failed for {label} with code {out.returncode}')

    print('[ok]', label)



run_leak(BASE_PLANNER, LEAK_BASE_LABEL)

run_leak(FT_PLANNER, LEAK_FT_LABEL)

run_leak(str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), LEAK_NULL_LABEL)



def load_leak_ms(label: str):

    p = REPO_ROOT / 'runs' / label / 'multi_seed.json'

    obj = json.loads(p.read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}



lb = load_leak_ms(LEAK_BASE_LABEL)

lf = load_leak_ms(LEAK_FT_LABEL)

ln = load_leak_ms(LEAK_NULL_LABEL)

pols = sorted(set(lb) & set(lf) & set(ln))

lfs = [lf[p]['task_success_rate']['mean'] - lb[p]['task_success_rate']['mean'] for p in pols]

lfi = [lf[p]['invalid_tool_call_rate']['mean'] - lb[p]['invalid_tool_call_rate']['mean'] for p in pols]

lns = [ln[p]['task_success_rate']['mean'] - lb[p]['task_success_rate']['mean'] for p in pols]

lni = [ln[p]['invalid_tool_call_rate']['mean'] - lb[p]['invalid_tool_call_rate']['mean'] for p in pols]

leak_ft_s = sum(lfs) / len(lfs) if lfs else 0.0

leak_ft_i = sum(lfi) / len(lfi) if lfi else 0.0

leak_null_s = sum(lns) / len(lns) if lns else 0.0

leak_null_i = sum(lni) / len(lni) if lni else 0.0

leak_adv_s = leak_ft_s - leak_null_s

leak_adv_i = leak_ft_i - leak_null_i

leak_invariant = abs(leak_adv_s) < 1e-6 and abs(leak_adv_i) < 1e-6



print('\n=== Leak-Probe Eval Summary ===')

print('toolset_id =', LEAK_TOOLSET_ID)

print('policies =', pols)

print(f'ft success delta: {leak_ft_s:+.4f}')

print(f'ft invalid delta: {leak_ft_i:+.4f}')

print(f'null success delta: {leak_null_s:+.4f}')

print(f'null invalid delta: {leak_null_i:+.4f}')

print(f'adapter advantage success: {leak_adv_s:+.4f}')

print(f'adapter advantage invalid: {leak_adv_i:+.4f}')

print('invariant across ft/null vs base:', leak_invariant)
code
#VSC-b0636cba
python
from pathlib import Path

planner_dir = REPO_ROOT / 'configs' / 'planners'

available_planners = sorted(p.name for p in planner_dir.glob('*.json'))

print('Available planners:')

for name in available_planners:

    print('-', name)
code
#VSC-f1205f47
python
sync_cmd = ['git', 'pull', '--ff-only', 'origin', 'main']
print('[publish] Syncing repo:', ' '.join(sync_cmd))
sync_res = subprocess.run(sync_cmd, text=True, capture_output=True, cwd=str(REPO_ROOT), check=False)
if sync_res.stdout:
    print(sync_res.stdout)
if sync_res.returncode != 0:
    if sync_res.stderr:
        print(sync_res.stderr)
    raise RuntimeError(f'Publish sync failed with code {sync_res.returncode}')

publish_script = REPO_ROOT / 'scripts' / 'publish_kaggle_northstar_artifacts.py'

if not publish_script.exists():

    print('[publish] helper script not present in this runtime clone; skipping publish step.')

else:

    dry_run_cmd = [

        'uv', 'run', 'python', str(publish_script),

        '--dataset-slug', PUBLISH_DATASET_SLUG,

        '--title', PUBLISH_DATASET_TITLE,

        '--label-prefix', LABEL_PREFIX,

        '--repo-root', '.',

        '--dry-run',

    ]

    print('[publish] Preflight dry-run:', ' '.join(dry_run_cmd))

    dry_res = subprocess.run(dry_run_cmd, text=True, capture_output=True, check=False)

    if dry_res.stdout:

        print(dry_res.stdout)

    if dry_res.returncode != 0:

        if dry_res.stderr:

            print(dry_res.stderr)

        raise RuntimeError(f'Publish preflight failed with code {dry_res.returncode}')

    publish_cmd = [

        'uv', 'run', 'python', str(publish_script),

        '--dataset-slug', PUBLISH_DATASET_SLUG,

        '--title', PUBLISH_DATASET_TITLE,

        '--label-prefix', LABEL_PREFIX,

        '--repo-root', '.',

    ]

    print('Running:', ' '.join(publish_cmd))

    publish_res = subprocess.run(publish_cmd, text=True, capture_output=True, check=False)

    if publish_res.stdout:

        print(publish_res.stdout)

    if publish_res.returncode != 0:

        if publish_res.stderr:

            print(publish_res.stderr)

        raise RuntimeError(f'Publish failed with code {publish_res.returncode}')
code
#VSC-2ae76f27
python
from pathlib import Path


import json




run_root = REPO_ROOT / 'runs'






def load_policy_metrics(ms_path: Path) -> dict[str, dict]:


    obj = json.loads(ms_path.read_text(encoding='utf-8'))


    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}






def mean_delta(base_ms: Path, ft_ms: Path) -> tuple[float, float] | None:


    if not base_ms.exists() or not ft_ms.exists():


        return None


    base_by = load_policy_metrics(base_ms)


    ft_by = load_policy_metrics(ft_ms)


    policies = sorted(set(base_by) & set(ft_by))


    if not policies:


        return 0.0, 0.0


    succ = [


        ft_by[p]['task_success_rate']['mean'] - base_by[p]['task_success_rate']['mean']


        for p in policies


    ]


    inv = [


        ft_by[p]['invalid_tool_call_rate']['mean'] - base_by[p]['invalid_tool_call_rate']['mean']


        for p in policies


    ]


    return sum(succ) / len(succ), sum(inv) / len(inv)






def matrix_verdict(matrix_path: Path) -> str:


    if not matrix_path.exists():


        return 'MISSING'


    obj = json.loads(matrix_path.read_text(encoding='utf-8'))


    return str(obj.get('portfolio_verdict', 'UNKNOWN'))






def triad_adv(base_ms: Path, ft_ms: Path, null_ms: Path) -> tuple[float, float] | None:


    if not (base_ms.exists() and ft_ms.exists() and null_ms.exists()):


        return None


    b = load_policy_metrics(base_ms)


    f = load_policy_metrics(ft_ms)


    n = load_policy_metrics(null_ms)


    policies = sorted(set(b) & set(f) & set(n))


    if not policies:


        return 0.0, 0.0


    ft_s = sum(f[p]['task_success_rate']['mean'] - b[p]['task_success_rate']['mean'] for p in policies) / len(policies)


    ft_i = sum(f[p]['invalid_tool_call_rate']['mean'] - b[p]['invalid_tool_call_rate']['mean'] for p in policies) / len(policies)


    n_s = sum(n[p]['task_success_rate']['mean'] - b[p]['task_success_rate']['mean'] for p in policies) / len(policies)


    n_i = sum(n[p]['invalid_tool_call_rate']['mean'] - b[p]['invalid_tool_call_rate']['mean'] for p in policies) / len(policies)


    return ft_s - n_s, ft_i - n_i






def ms_path(label: str) -> Path:


    return run_root / label / 'multi_seed.json'






def matrix_path(label: str) -> Path:


    return run_root / label / 'matrix.json'






rows = []




# Core runs


core = [


    ('sensitivity', LABEL_PREFIX),


    ('recovery', f'{LABEL_PREFIX}-recovery'),


    ('calibration', 'toolsetrel-hf-qwen25-3b-calibration-v1'),


    ('enriched_cs', f'{LABEL_PREFIX}-enriched-cs'),


]




for name, prefix in core:


    d = mean_delta(ms_path(f'{prefix}-base-ms'), ms_path(f'{prefix}-ft-ms'))


    v = matrix_verdict(matrix_path(f'{prefix}-matrix'))


    if d is None:


        rows.append((name, 'MISSING', 'MISSING', v))


    else:


        rows.append((name, f'{d[0]:+.4f}', f'{d[1]:+.4f}', v))




# Model-sensitive per toolset


for toolset in ['customer_support', 'ecommerce_ops', 'fintech_risk']:


    prefix = f'{LABEL_PREFIX}-msonly-{toolset}'


    d = mean_delta(ms_path(f'{prefix}-base'), ms_path(f'{prefix}-ft'))


    adv = triad_adv(ms_path(f'{prefix}-base'), ms_path(f'{prefix}-ft'), ms_path(f'{prefix}-null'))


    if d is None or adv is None:


        rows.append((f'msonly_{toolset}', 'MISSING', 'MISSING', 'N/A'))


    else:


        rows.append((f'msonly_{toolset}', f'{d[0]:+.4f}', f'{d[1]:+.4f}', f'adv=({adv[0]:+.4f},{adv[1]:+.4f})'))




# Synthetic/leak triads


for name, prefix in [


    ('synthetic_ambiguous', f'{LABEL_PREFIX}-syn-ambiguous'),


    ('leak_probe', f'{LABEL_PREFIX}-leakprobe'),


]:


    adv = triad_adv(ms_path(f'{prefix}-base'), ms_path(f'{prefix}-ft'), ms_path(f'{prefix}-null'))


    if adv is None:


        rows.append((name, 'MISSING', 'MISSING', 'N/A'))


    else:


        d = mean_delta(ms_path(f'{prefix}-base'), ms_path(f'{prefix}-ft'))


        rows.append((name, f'{d[0]:+.4f}', f'{d[1]:+.4f}', f'adv=({adv[0]:+.4f},{adv[1]:+.4f})'))




print('=== Decision Dashboard (Artifact Summary) ===')


print('experiment | mean_success_delta | mean_invalid_delta | verdict_or_adv')


for r in rows:


    print(' | '.join(r))
markdown
#VSC-026c1861
markdown
## Ready For Execution

Run cells top-to-bottom in Kaggle runtime to execute and publish the Toolset Reliability sensitivity study.
markdown
#VSC-20869eb9
markdown
## Fast-Signal Implementation Path (24h, Low Compute)

This compact path is the primary implementation target for the current cycle.

Execution intent:
- Run one model-sensitive enriched triad (base vs finetuned vs null-adapter)
- Confirm non-flat signal and adapter advantage over null control
- Run one lightweight northstar check using the same setup

Fast-signal acceptance criteria (pre-registered):
- mean success delta (ft-base) > 0.0
- mean invalid delta (ft-base) <= 0.0
- adapter advantage vs null on success > 0.003
- adapter advantage vs null on invalid <= 0.0
- non-flat check: max absolute primary delta > 1e-4
code
#VSC-6e85ed34
python
import json
import subprocess
from pathlib import Path

FAST_WORKLOAD = 'workloads/enriched/customer_support.json'
FAST_SEEDS = '11,22,33'
FAST_LABEL_PREFIX = f"{LABEL_PREFIX}-fastsig-cs-v1"
FAST_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_fast_signal_cs_v1.json'
FAST_NULL_PLANNER = REPO_ROOT / 'configs' / 'planners' / 'hf_qwen2_5_3b_ft_nulladapter.json'

fast_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
fast_cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']
fast_faults = dict(fast_cfg.get('fault_probabilities', {}))
fast_faults['malformed_schema'] = max(float(fast_faults.get('malformed_schema', 0.06)), 0.10)
fast_faults['timeout'] = max(float(fast_faults.get('timeout', 0.08)), 0.10)
fast_cfg['fault_probabilities'] = fast_faults
fast_cfg['max_attempts'] = max(int(fast_cfg.get('max_attempts', 4)), 5)
fast_cfg['time_budget_ms'] = max(int(fast_cfg.get('time_budget_ms', 1800)), 2200)

FAST_CONFIG.write_text(json.dumps(fast_cfg, indent=2) + '\n', encoding='utf-8')

if not FAST_NULL_PLANNER.exists():
    null_obj = {
        'type': 'hf_local',
        'name': 'hf_qwen2_5_3b_ft_nulladapter',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
    }
    FAST_NULL_PLANNER.write_text(json.dumps(null_obj, indent=2) + '\n', encoding='utf-8')

print('FAST_CONFIG =', FAST_CONFIG)
print('FAST_WORKLOAD =', FAST_WORKLOAD)
print('FAST_SEEDS =', FAST_SEEDS)
print('FAST policies =', fast_cfg['policies'])
print('FAST faults =', fast_cfg['fault_probabilities'])
print('FAST_NULL_PLANNER =', FAST_NULL_PLANNER)
code
#VSC-c606876d
python
def run_multi_seed_fast(workload: str, planner_cfg: str, label: str) -> None:
    cmd = [
        'uv', 'run', 'tcrb', 'multi-seed',
        '--config', str(FAST_CONFIG),
        '--workload', workload,
        '--seeds', FAST_SEEDS,
        '--planner-config', planner_cfg,
        '--label', label,
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'fast multi-seed failed for {label} with code {rc}')


def load_policy_means_fast(label: str) -> dict:
    payload_path = REPO_ROOT / 'runs' / label / 'multi_seed.json'
    payload = json.loads(payload_path.read_text(encoding='utf-8'))
    return {r['policy']: r['metrics'] for r in payload.get('aggregate_policy_metrics', [])}


fast_base_label = f'{FAST_LABEL_PREFIX}-base'
fast_ft_label = f'{FAST_LABEL_PREFIX}-ft'
fast_null_label = f'{FAST_LABEL_PREFIX}-null'

run_multi_seed_fast(FAST_WORKLOAD, BASE_PLANNER, fast_base_label)
run_multi_seed_fast(FAST_WORKLOAD, FT_PLANNER, fast_ft_label)
run_multi_seed_fast(FAST_WORKLOAD, str(FAST_NULL_PLANNER.relative_to(REPO_ROOT)), fast_null_label)

fast_base = load_policy_means_fast(fast_base_label)
fast_ft = load_policy_means_fast(fast_ft_label)
fast_null = load_policy_means_fast(fast_null_label)
fast_policies = sorted(set(fast_base) & set(fast_ft) & set(fast_null))

ft_s = [fast_ft[p]['task_success_rate']['mean'] - fast_base[p]['task_success_rate']['mean'] for p in fast_policies]
ft_i = [fast_ft[p]['invalid_tool_call_rate']['mean'] - fast_base[p]['invalid_tool_call_rate']['mean'] for p in fast_policies]
null_s = [fast_null[p]['task_success_rate']['mean'] - fast_base[p]['task_success_rate']['mean'] for p in fast_policies]
null_i = [fast_null[p]['invalid_tool_call_rate']['mean'] - fast_base[p]['invalid_tool_call_rate']['mean'] for p in fast_policies]

fast_mean_success_delta = sum(ft_s) / len(ft_s) if ft_s else 0.0
fast_mean_invalid_delta = sum(ft_i) / len(ft_i) if ft_i else 0.0
fast_null_success_delta = sum(null_s) / len(null_s) if null_s else 0.0
fast_null_invalid_delta = sum(null_i) / len(null_i) if null_i else 0.0

fast_adapter_adv_success = fast_mean_success_delta - fast_null_success_delta
fast_adapter_adv_invalid = fast_mean_invalid_delta - fast_null_invalid_delta
fast_nonflat = max(
    abs(fast_mean_success_delta),
    abs(fast_mean_invalid_delta),
    abs(fast_adapter_adv_success),
    abs(fast_adapter_adv_invalid),
) > 1e-4

fast_signal_pass = (
    (fast_mean_success_delta > 0.0)
    and (fast_mean_invalid_delta <= 0.0)
    and (fast_adapter_adv_success > 0.003)
    and (fast_adapter_adv_invalid <= 0.0)
    and fast_nonflat
)

FAST_TRIAD_SUMMARY = {
    'label_prefix': FAST_LABEL_PREFIX,
    'workload': FAST_WORKLOAD,
    'seeds': FAST_SEEDS,
    'policies': fast_policies,
    'ft_base_success_delta': fast_mean_success_delta,
    'ft_base_invalid_delta': fast_mean_invalid_delta,
    'null_base_success_delta': fast_null_success_delta,
    'null_base_invalid_delta': fast_null_invalid_delta,
    'adapter_adv_success': fast_adapter_adv_success,
    'adapter_adv_invalid': fast_adapter_adv_invalid,
    'nonflat': fast_nonflat,
    'fast_signal_pass': fast_signal_pass,
}

print('\n=== Fast Triad Summary ===')
print(json.dumps(FAST_TRIAD_SUMMARY, indent=2))
code
#VSC-215160fe
python
FAST_NS_LABEL_PREFIX = f'{FAST_LABEL_PREFIX}-northstar'

northstar_cmd = [
    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',
    '--config', str(FAST_CONFIG),
    '--workload', FAST_WORKLOAD,
    '--seeds', FAST_SEEDS,
    '--base-planner-config', BASE_PLANNER,
    '--ft-planner-config', FT_PLANNER,
    '--label-prefix', FAST_NS_LABEL_PREFIX,
]

print('Running:', ' '.join(northstar_cmd))
ns_proc = subprocess.Popen(northstar_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert ns_proc.stdout is not None
for line in ns_proc.stdout:
    print(line, end='')
ns_rc = ns_proc.wait()
if ns_rc != 0:
    raise RuntimeError(f'fast northstar run failed with code {ns_rc}')

run_root = REPO_ROOT / 'runs'
ns_base_ms = run_root / f'{FAST_NS_LABEL_PREFIX}-base-ms' / 'multi_seed.json'
ns_ft_ms = run_root / f'{FAST_NS_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'
ns_matrix_json = run_root / f'{FAST_NS_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [ns_base_ms, ns_ft_ms, ns_matrix_json]:
    print('-', p, 'exists=' + str(p.exists()))

ns_gate_dir = run_root / f'{FAST_NS_LABEL_PREFIX}-study-gate'
ns_gate_dir.mkdir(parents=True, exist_ok=True)
ns_gate_json = ns_gate_dir / 'study_gate.json'
ns_gate_md = ns_gate_dir / 'study_gate.md'

gate_cmd = [
    'uv', 'run', 'python', '-m', 'tcrb', 'study-gate',
    '--base-run', str(ns_base_ms),
    '--finetuned-run', str(ns_ft_ms),
    '--matrix-json', str(ns_matrix_json),
    '--require-matrix-signal',
    '--require-matrix-not-fail',
    '--output-json', str(ns_gate_json),
    '--output-report', str(ns_gate_md),
]

print('Running:', ' '.join(gate_cmd))
gate_res = subprocess.run(gate_cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)
if gate_res.stdout:
    print(gate_res.stdout)
if gate_res.returncode != 0 and gate_res.stderr:
    print(gate_res.stderr)

ns_matrix_obj = json.loads(ns_matrix_json.read_text(encoding='utf-8'))
ns_gate_obj = json.loads(ns_gate_json.read_text(encoding='utf-8')) if ns_gate_json.exists() else {}

ns_rows = ns_matrix_obj.get('rows', [])
ns_max_abs_matrix_delta = max(
    [
        abs(float(r.get('delta_first_tool_accuracy', 0.0)))
        for r in ns_rows
    ]
    + [
        abs(float(r.get('delta_sequence_prefix_accuracy', 0.0)))
        for r in ns_rows
    ]
    + [0.0]
)

ns_study_verdict = str(ns_gate_obj.get('verdict', 'MISSING'))
ns_matrix_verdict = str(ns_matrix_obj.get('portfolio_verdict', 'MISSING'))
if ns_study_verdict == 'PASS':
    ns_fail_class = 'PASS'
elif ns_max_abs_matrix_delta <= 1e-4:
    ns_fail_class = 'STRUCTURAL_FLAT_FAIL'
else:
    ns_fail_class = 'THRESHOLD_TIGHT_FAIL'

FAST_NORTHSTAR_SUMMARY = {
    'label_prefix': FAST_NS_LABEL_PREFIX,
    'matrix_portfolio_verdict': ns_matrix_verdict,
    'study_gate_verdict': ns_study_verdict,
    'matrix_max_abs_delta': ns_max_abs_matrix_delta,
    'failure_classification': ns_fail_class,
    'gate_checks': ns_gate_obj.get('checks', []),
}

print('\n=== Fast Northstar Summary ===')
print(json.dumps(FAST_NORTHSTAR_SUMMARY, indent=2))
code
#VSC-033556b7
python
print('=== Fast Signal Final Decision ===')
print('triad_fast_signal_pass =', FAST_TRIAD_SUMMARY.get('fast_signal_pass'))
print('northstar_failure_classification =', FAST_NORTHSTAR_SUMMARY.get('failure_classification'))

if FAST_TRIAD_SUMMARY.get('fast_signal_pass'):
    if FAST_NORTHSTAR_SUMMARY.get('failure_classification') == 'PASS':
        decision = 'GO: meaningful signal found and northstar-lite passed.'
    else:
        decision = 'GO-PHASE-2: meaningful signal found; proceed to northstar-grade portfolio run.'
else:
    if FAST_NORTHSTAR_SUMMARY.get('failure_classification') == 'STRUCTURAL_FLAT_FAIL':
        decision = 'PIVOT: benchmark still flat; refresh finetune data before more benchmarking.'
    else:
        decision = 'HOLD: weak signal; run one targeted hard-slice triad before retraining.'

print('decision =', decision)

FAST_DECISION_SUMMARY = {
    'triad': FAST_TRIAD_SUMMARY,
    'northstar': FAST_NORTHSTAR_SUMMARY,
    'decision': decision,
}
print('\nFAST_DECISION_SUMMARY_JSON')
print(json.dumps(FAST_DECISION_SUMMARY, indent=2))

[deps] Probe output: MISSING=torch,transformers,peft,trl,datasets,accelerate,bitsandbytes,wrapt
Running: uv pip install --python .venv/bin/python torch transformers peft trl datasets accelerate bitsandbytes wrapt
[deps] uv environment dependencies are ready.


## Fast-Signal (Kaggle Kernel Execution)

This block runs only in the attached notebook kernel (Kaggle GPU) and avoids local heavy execution.


In [40]:
import json
import subprocess
from pathlib import Path

FAST_WORKLOAD = 'workloads/enriched/customer_support.json'
FAST_SEEDS = '11,22,33'
FAST_LABEL_PREFIX = f"{LABEL_PREFIX}-fastsig-cs-v1-kernel"
FAST_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_fast_signal_cs_v1.json'
FAST_NULL_PLANNER = REPO_ROOT / 'configs' / 'planners' / 'hf_qwen2_5_3b_ft_nulladapter.json'

fast_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
fast_cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']
fast_faults = dict(fast_cfg.get('fault_probabilities', {}))
fast_faults['malformed_schema'] = max(float(fast_faults.get('malformed_schema', 0.06)), 0.10)
fast_faults['timeout'] = max(float(fast_faults.get('timeout', 0.08)), 0.10)
fast_cfg['fault_probabilities'] = fast_faults
fast_cfg['max_attempts'] = max(int(fast_cfg.get('max_attempts', 4)), 5)
fast_cfg['time_budget_ms'] = max(int(fast_cfg.get('time_budget_ms', 1800)), 2200)
FAST_CONFIG.write_text(json.dumps(fast_cfg, indent=2) + '\n', encoding='utf-8')

if not FAST_NULL_PLANNER.exists():
    null_obj = {
        'type': 'hf_local',
        'name': 'hf_qwen2_5_3b_ft_nulladapter',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
    }
    FAST_NULL_PLANNER.write_text(json.dumps(null_obj, indent=2) + '\n', encoding='utf-8')

print('FAST_CONFIG =', FAST_CONFIG)
print('FAST_WORKLOAD =', FAST_WORKLOAD)
print('FAST_SEEDS =', FAST_SEEDS)
print('FAST_LABEL_PREFIX =', FAST_LABEL_PREFIX)
print('FAST_NULL_PLANNER =', FAST_NULL_PLANNER)

FAST_CONFIG = /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json
FAST_WORKLOAD = workloads/enriched/customer_support.json
FAST_SEEDS = 11,22,33
FAST_LABEL_PREFIX = toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel
FAST_NULL_PLANNER = /kaggle/working/tool-calling-reliability-benchmark/configs/planners/hf_qwen2_5_3b_ft_nulladapter.json


In [41]:
def run_multi_seed_fast(workload: str, planner_cfg: str, label: str) -> None:
    cmd = [
        'uv', 'run', 'tcrb', 'multi-seed',
        '--config', str(FAST_CONFIG),
        '--workload', workload,
        '--seeds', FAST_SEEDS,
        '--planner-config', planner_cfg,
        '--label', label,
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'fast multi-seed failed for {label} with code {rc}')


def load_policy_means_fast(label: str) -> dict:
    payload_path = REPO_ROOT / 'runs' / label / 'multi_seed.json'
    payload = json.loads(payload_path.read_text(encoding='utf-8'))
    return {r['policy']: r['metrics'] for r in payload.get('aggregate_policy_metrics', [])}


fast_base_label = f'{FAST_LABEL_PREFIX}-base'
fast_ft_label = f'{FAST_LABEL_PREFIX}-ft'
fast_null_label = f'{FAST_LABEL_PREFIX}-null'

run_multi_seed_fast(FAST_WORKLOAD, BASE_PLANNER, fast_base_label)
run_multi_seed_fast(FAST_WORKLOAD, FT_PLANNER, fast_ft_label)
run_multi_seed_fast(FAST_WORKLOAD, str(FAST_NULL_PLANNER.relative_to(REPO_ROOT)), fast_null_label)

fast_base = load_policy_means_fast(fast_base_label)
fast_ft = load_policy_means_fast(fast_ft_label)
fast_null = load_policy_means_fast(fast_null_label)
fast_policies = sorted(set(fast_base) & set(fast_ft) & set(fast_null))

ft_s = [fast_ft[p]['task_success_rate']['mean'] - fast_base[p]['task_success_rate']['mean'] for p in fast_policies]
ft_i = [fast_ft[p]['invalid_tool_call_rate']['mean'] - fast_base[p]['invalid_tool_call_rate']['mean'] for p in fast_policies]
null_s = [fast_null[p]['task_success_rate']['mean'] - fast_base[p]['task_success_rate']['mean'] for p in fast_policies]
null_i = [fast_null[p]['invalid_tool_call_rate']['mean'] - fast_base[p]['invalid_tool_call_rate']['mean'] for p in fast_policies]

fast_mean_success_delta = sum(ft_s) / len(ft_s) if ft_s else 0.0
fast_mean_invalid_delta = sum(ft_i) / len(ft_i) if ft_i else 0.0
fast_null_success_delta = sum(null_s) / len(null_s) if null_s else 0.0
fast_null_invalid_delta = sum(null_i) / len(null_i) if null_i else 0.0

fast_adapter_adv_success = fast_mean_success_delta - fast_null_success_delta
fast_adapter_adv_invalid = fast_mean_invalid_delta - fast_null_invalid_delta
fast_nonflat = max(
    abs(fast_mean_success_delta),
    abs(fast_mean_invalid_delta),
    abs(fast_adapter_adv_success),
    abs(fast_adapter_adv_invalid),
) > 1e-4

fast_signal_pass = (
    (fast_mean_success_delta > 0.0)
    and (fast_mean_invalid_delta <= 0.0)
    and (fast_adapter_adv_success > 0.003)
    and (fast_adapter_adv_invalid <= 0.0)
    and fast_nonflat
)

FAST_TRIAD_SUMMARY = {
    'label_prefix': FAST_LABEL_PREFIX,
    'workload': FAST_WORKLOAD,
    'seeds': FAST_SEEDS,
    'policies': fast_policies,
    'ft_base_success_delta': fast_mean_success_delta,
    'ft_base_invalid_delta': fast_mean_invalid_delta,
    'null_base_success_delta': fast_null_success_delta,
    'null_base_invalid_delta': fast_null_invalid_delta,
    'adapter_adv_success': fast_adapter_adv_success,
    'adapter_adv_invalid': fast_adapter_adv_invalid,
    'nonflat': fast_nonflat,
    'fast_signal_pass': fast_signal_pass,
}

print('\n=== Fast Triad Summary ===')
print(json.dumps(FAST_TRIAD_SUMMARY, indent=2))

Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-base

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 127.08it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-base/multi_seed.json
Wrote multi-seed summary: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-base/multi_seed_summary.md
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_ft.json --label toolsetrel-hf-kaggle-qwen25-3b-se

In [43]:
FAST_NS_LABEL_PREFIX = f'{FAST_LABEL_PREFIX}-northstar'

northstar_cmd = [
    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',
    '--config', str(FAST_CONFIG),
    '--workload', FAST_WORKLOAD,
    '--seeds', FAST_SEEDS,
    '--base-planner-config', BASE_PLANNER,
    '--ft-planner-config', FT_PLANNER,
    '--label-prefix', FAST_NS_LABEL_PREFIX,
]

print('Running:', ' '.join(northstar_cmd))
ns_proc = subprocess.Popen(northstar_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert ns_proc.stdout is not None
for line in ns_proc.stdout:
    print(line, end='')
ns_rc = ns_proc.wait()
if ns_rc != 0:
    raise RuntimeError(f'fast northstar run failed with code {ns_rc}')

run_root = REPO_ROOT / 'runs'
ns_base_ms = run_root / f'{FAST_NS_LABEL_PREFIX}-base-ms' / 'multi_seed.json'
ns_ft_ms = run_root / f'{FAST_NS_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'
ns_matrix_json = run_root / f'{FAST_NS_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [ns_base_ms, ns_ft_ms, ns_matrix_json]:
    print('-', p, 'exists=' + str(p.exists()))

ns_gate_dir = run_root / f'{FAST_NS_LABEL_PREFIX}-study-gate'
ns_gate_dir.mkdir(parents=True, exist_ok=True)
ns_gate_json = ns_gate_dir / 'study_gate.json'
ns_gate_md = ns_gate_dir / 'study_gate.md'

gate_cmd = [
    'uv', 'run', 'python', '-m', 'tcrb', 'study-gate',
    '--base-run', str(ns_base_ms),
    '--finetuned-run', str(ns_ft_ms),
    '--matrix-json', str(ns_matrix_json),
    '--require-matrix-signal',
    '--require-matrix-not-fail',
    '--output-json', str(ns_gate_json),
    '--output-report', str(ns_gate_md),
]

print('Running:', ' '.join(gate_cmd))
gate_res = subprocess.run(gate_cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)
if gate_res.stdout:
    print(gate_res.stdout)
if gate_res.returncode != 0 and gate_res.stderr:
    print(gate_res.stderr)

ns_matrix_obj = json.loads(ns_matrix_json.read_text(encoding='utf-8'))
ns_gate_obj = json.loads(ns_gate_json.read_text(encoding='utf-8')) if ns_gate_json.exists() else {}

ns_rows = ns_matrix_obj.get('rows', [])
ns_max_abs_matrix_delta = max(
    [abs(float(r.get('delta_first_tool_accuracy', 0.0))) for r in ns_rows]
    + [abs(float(r.get('delta_sequence_prefix_accuracy', 0.0))) for r in ns_rows]
    + [0.0]
)

ns_study_verdict = str(ns_gate_obj.get('verdict', 'MISSING'))
ns_matrix_verdict = str(ns_matrix_obj.get('portfolio_verdict', 'MISSING'))
if ns_study_verdict == 'PASS':
    ns_fail_class = 'PASS'
elif ns_max_abs_matrix_delta <= 1e-4:
    ns_fail_class = 'STRUCTURAL_FLAT_FAIL'
else:
    ns_fail_class = 'THRESHOLD_TIGHT_FAIL'

FAST_NORTHSTAR_SUMMARY = {
    'label_prefix': FAST_NS_LABEL_PREFIX,
    'matrix_portfolio_verdict': ns_matrix_verdict,
    'study_gate_verdict': ns_study_verdict,
    'matrix_max_abs_delta': ns_max_abs_matrix_delta,
    'failure_classification': ns_fail_class,
    'gate_checks': ns_gate_obj.get('checks', []),
}

if FAST_TRIAD_SUMMARY.get('fast_signal_pass'):
    if FAST_NORTHSTAR_SUMMARY.get('failure_classification') == 'PASS':
        decision = 'GO: meaningful signal found and northstar-lite passed.'
    else:
        decision = 'GO-PHASE-2: meaningful signal found; proceed to northstar-grade portfolio run.'
else:
    if FAST_NORTHSTAR_SUMMARY.get('failure_classification') == 'STRUCTURAL_FLAT_FAIL':
        decision = 'PIVOT: benchmark still flat; refresh finetune data before more benchmarking.'
    else:
        decision = 'HOLD: weak signal; run one targeted hard-slice triad before retraining.'

FAST_DECISION_SUMMARY = {
    'triad': FAST_TRIAD_SUMMARY,
    'northstar': FAST_NORTHSTAR_SUMMARY,
    'decision': decision,
}

print('\n=== Fast Northstar Summary ===')
print(json.dumps(FAST_NORTHSTAR_SUMMARY, indent=2))
print('\n=== Fast Decision ===')
print(decision)
print('\nFAST_DECISION_SUMMARY_JSON')
print(json.dumps(FAST_DECISION_SUMMARY, indent=2))

Running: uv run python scripts/run_northstar_hf.py --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft.json --label-prefix toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-northstar
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-northstar-base-ms

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 126.44it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-

In [42]:
print('FAST_TRIAD_SUMMARY_JSON')
print(json.dumps(FAST_TRIAD_SUMMARY, indent=2))

FAST_TRIAD_SUMMARY_JSON
{
  "label_prefix": "toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel",
  "workload": "workloads/enriched/customer_support.json",
  "seeds": "11,22,33",
  "policies": [
    "exponential_backoff_jitter",
    "naive_retry"
  ],
  "ft_base_success_delta": 0.0,
  "ft_base_invalid_delta": 0.0,
  "null_base_success_delta": 0.0,
  "null_base_invalid_delta": 0.0,
  "adapter_adv_success": 0.0,
  "adapter_adv_invalid": 0.0,
  "nonflat": false,
  "fast_signal_pass": false
}


In [44]:
print('FAST_NORTHSTAR_SUMMARY_JSON')
print(json.dumps(FAST_NORTHSTAR_SUMMARY, indent=2))
print('FAST_DECISION =', FAST_DECISION_SUMMARY.get('decision'))

FAST_NORTHSTAR_SUMMARY_JSON
{
  "label_prefix": "toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-northstar",
  "matrix_portfolio_verdict": "FAIL",
  "study_gate_verdict": "FAIL",
  "matrix_max_abs_delta": 0.02777777777777779,
  "failure_classification": "THRESHOLD_TIGHT_FAIL",
  "gate_checks": [
    {
      "name": "base_vs_ft_nonflatline",
      "passed": false,
      "value": 0.0,
      "threshold": 0.0001,
      "detail": "max abs core delta must exceed flatline epsilon"
    },
    {
      "name": "matrix_nonflatline",
      "passed": true,
      "value": 0.02777777777777779,
      "threshold": 0.0001,
      "detail": "max abs transfer-matrix delta must exceed epsilon"
    },
    {
      "name": "matrix_portfolio_not_fail",
      "passed": false,
      "value": "FAIL",
      "threshold": "not FAIL",
      "detail": "transfer-matrix portfolio verdict must not be FAIL"
    }
  ]
}
FAST_DECISION = HOLD: weak signal; run one targeted hard-slice triad before retraining

## Targeted Hard-Slice Triad (Kaggle Kernel)

Runs one focused triad on a hard slice (prefer leak-probe, fallback synthetic) to decide whether to continue benchmarking or pivot to data refresh + retrain.

In [45]:
import json
import subprocess

# Prefer the leak-probe slice as the targeted hard slice; fall back to synthetic slice.
_hard_candidate = None
_hard_tag = None
if 'LEAK_WORKLOAD_PATH' in globals() and LEAK_WORKLOAD_PATH.exists():
    _hard_candidate = LEAK_WORKLOAD_PATH
    _hard_tag = 'leak-probe'
elif 'SYN_WORKLOAD_PATH' in globals() and SYN_WORKLOAD_PATH.exists():
    _hard_candidate = SYN_WORKLOAD_PATH
    _hard_tag = 'synthetic-ambiguous'
else:
    _hard_candidate = REPO_ROOT / FAST_WORKLOAD
    _hard_tag = 'customer-support-fallback'

HARD_SLICE_WORKLOAD = str(_hard_candidate.relative_to(REPO_ROOT))
HARD_SLICE_LABEL_PREFIX = f"{FAST_LABEL_PREFIX}-hard-{_hard_tag}"


def run_hard_ms(planner_cfg: str, label: str) -> None:
    cmd = [
        'uv', 'run', 'tcrb', 'multi-seed',
        '--config', str(FAST_CONFIG),
        '--workload', HARD_SLICE_WORKLOAD,
        '--seeds', FAST_SEEDS,
        '--planner-config', planner_cfg,
        '--label', label,
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'hard-slice multi-seed failed for {label} with code {rc}')


def load_ms_policy_means(label: str) -> dict:
    payload_path = REPO_ROOT / 'runs' / label / 'multi_seed.json'
    payload = json.loads(payload_path.read_text(encoding='utf-8'))
    return {r['policy']: r['metrics'] for r in payload.get('aggregate_policy_metrics', [])}


hard_base_label = f'{HARD_SLICE_LABEL_PREFIX}-base'
hard_ft_label = f'{HARD_SLICE_LABEL_PREFIX}-ft'
hard_null_label = f'{HARD_SLICE_LABEL_PREFIX}-null'

run_hard_ms(BASE_PLANNER, hard_base_label)
run_hard_ms(FT_PLANNER, hard_ft_label)
run_hard_ms(str(FAST_NULL_PLANNER.relative_to(REPO_ROOT)), hard_null_label)

hard_base = load_ms_policy_means(hard_base_label)
hard_ft = load_ms_policy_means(hard_ft_label)
hard_null = load_ms_policy_means(hard_null_label)

hard_policies = sorted(set(hard_base) & set(hard_ft) & set(hard_null))

hard_ft_s = [hard_ft[p]['task_success_rate']['mean'] - hard_base[p]['task_success_rate']['mean'] for p in hard_policies]
hard_ft_i = [hard_ft[p]['invalid_tool_call_rate']['mean'] - hard_base[p]['invalid_tool_call_rate']['mean'] for p in hard_policies]
hard_null_s = [hard_null[p]['task_success_rate']['mean'] - hard_base[p]['task_success_rate']['mean'] for p in hard_policies]
hard_null_i = [hard_null[p]['invalid_tool_call_rate']['mean'] - hard_base[p]['invalid_tool_call_rate']['mean'] for p in hard_policies]

hard_ft_success_delta = sum(hard_ft_s) / len(hard_ft_s) if hard_ft_s else 0.0
hard_ft_invalid_delta = sum(hard_ft_i) / len(hard_ft_i) if hard_ft_i else 0.0
hard_null_success_delta = sum(hard_null_s) / len(hard_null_s) if hard_null_s else 0.0
hard_null_invalid_delta = sum(hard_null_i) / len(hard_null_i) if hard_null_i else 0.0

hard_adapter_adv_success = hard_ft_success_delta - hard_null_success_delta
hard_adapter_adv_invalid = hard_ft_invalid_delta - hard_null_invalid_delta
hard_nonflat = max(
    abs(hard_ft_success_delta),
    abs(hard_ft_invalid_delta),
    abs(hard_adapter_adv_success),
    abs(hard_adapter_adv_invalid),
) > 1e-4

hard_signal_pass = (
    (hard_ft_success_delta > 0.0)
    and (hard_ft_invalid_delta <= 0.0)
    and (hard_adapter_adv_success > 0.003)
    and (hard_adapter_adv_invalid <= 0.0)
    and hard_nonflat
)

HARD_SLICE_TRIAD_SUMMARY = {
    'label_prefix': HARD_SLICE_LABEL_PREFIX,
    'slice_tag': _hard_tag,
    'workload': HARD_SLICE_WORKLOAD,
    'seeds': FAST_SEEDS,
    'policies': hard_policies,
    'ft_base_success_delta': hard_ft_success_delta,
    'ft_base_invalid_delta': hard_ft_invalid_delta,
    'null_base_success_delta': hard_null_success_delta,
    'null_base_invalid_delta': hard_null_invalid_delta,
    'adapter_adv_success': hard_adapter_adv_success,
    'adapter_adv_invalid': hard_adapter_adv_invalid,
    'nonflat': hard_nonflat,
    'hard_signal_pass': hard_signal_pass,
}

if hard_signal_pass:
    hard_decision = 'GO-PHASE-2: hard-slice triad found meaningful adapter signal; proceed to northstar-grade run.'
else:
    hard_decision = 'PIVOT: hard-slice triad still weak/flat; refresh finetune data and retrain before more benchmarking.'

HARD_SLICE_DECISION = {
    'hard_slice': HARD_SLICE_TRIAD_SUMMARY,
    'decision': hard_decision,
}

print('\nHARD_SLICE_TRIAD_SUMMARY_JSON')
print(json.dumps(HARD_SLICE_TRIAD_SUMMARY, indent=2))
print('\nHARD_SLICE_DECISION =', hard_decision)

Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/ambiguous_ops_leak_probe.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-leak-probe-base

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 127.67it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-leak-probe-base/multi_seed.json
Wrote multi-seed summary: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-leak-probe-base/multi_seed_summary.md
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/ambiguous_ops_leak_probe.json --seeds 11,22,33 --planner-config configs/planners

In [46]:
print('HARD_SLICE_TRIAD_SUMMARY_JSON')
print(json.dumps(HARD_SLICE_TRIAD_SUMMARY, indent=2))
print('HARD_SLICE_DECISION =', HARD_SLICE_DECISION.get('decision'))
print('PRIOR_FAST_DECISION =', FAST_DECISION_SUMMARY.get('decision'))

HARD_SLICE_TRIAD_SUMMARY_JSON
{
  "label_prefix": "toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-leak-probe",
  "slice_tag": "leak-probe",
  "workload": "workloads/enriched/ambiguous_ops_leak_probe.json",
  "seeds": "11,22,33",
  "policies": [
    "exponential_backoff_jitter",
    "naive_retry"
  ],
  "ft_base_success_delta": 0.0,
  "ft_base_invalid_delta": 0.0,
  "null_base_success_delta": 0.0,
  "null_base_invalid_delta": 0.0,
  "adapter_adv_success": 0.0,
  "adapter_adv_invalid": 0.0,
  "nonflat": false,
  "hard_signal_pass": false
}
HARD_SLICE_DECISION = PIVOT: hard-slice triad still weak/flat; refresh finetune data and retrain before more benchmarking.
PRIOR_FAST_DECISION = HOLD: weak signal; run one targeted hard-slice triad before retraining.


## Data Refresh + Retrain + Fast Triad Recheck (Kaggle Kernel)

This section refreshes finetune data from new hard and baseline slices, retrains a QLoRA adapter in-kernel, and reruns the same fast triad for a before/after comparison.

In [47]:
import json
import subprocess
from pathlib import Path

REFRESH_TAG = 'hard-refresh-v1'
REFRESH_ROOT = REPO_ROOT / 'finetuned-models' / REFRESH_TAG
REFRESH_ROOT.mkdir(parents=True, exist_ok=True)

source_specs = [
    {
        'name': 'hard',
        'workload': HARD_SLICE_WORKLOAD,
        'label': f"{FAST_LABEL_PREFIX}-{REFRESH_TAG}-source-hard",
        'split_seed': '111',
    },
    {
        'name': 'baseline',
        'workload': FAST_WORKLOAD,
        'label': f"{FAST_LABEL_PREFIX}-{REFRESH_TAG}-source-baseline",
        'split_seed': '112',
    },
]


def _run_cmd(cmd: list[str]) -> None:
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"command failed with code {rc}: {' '.join(cmd)}")


for spec in source_specs:
    run_cmd = [
        'uv', 'run', 'tcrb', 'run',
        '--config', str(FAST_CONFIG),
        '--workload', spec['workload'],
        '--planner-config', BASE_PLANNER,
        '--label', spec['label'],
    ]
    _run_cmd(run_cmd)

    result_json = REPO_ROOT / 'runs' / spec['label'] / 'result.json'
    out_dir = REFRESH_ROOT / spec['name']
    out_dir.mkdir(parents=True, exist_ok=True)

    finetune_cmd = [
        'uv', 'run', 'tcrb', 'finetune-data',
        '--input-json', str(result_json),
        '--output-dir', str(out_dir),
        '--validation-split', '0.2',
        '--seed', spec['split_seed'],
        '--workload', spec['workload'],
        '--include-failure-attempts',
    ]
    _run_cmd(finetune_cmd)

MERGED_REFRESH_DIR = REFRESH_ROOT / 'merged'
MERGED_REFRESH_DIR.mkdir(parents=True, exist_ok=True)
REFRESH_TRAIN_JSONL = MERGED_REFRESH_DIR / 'train_dataset.jsonl'
REFRESH_EVAL_JSONL = MERGED_REFRESH_DIR / 'eval_dataset.jsonl'

seen_train = set()
seen_eval = set()
train_rows = 0
eval_rows = 0

with REFRESH_TRAIN_JSONL.open('w', encoding='utf-8') as train_out, REFRESH_EVAL_JSONL.open('w', encoding='utf-8') as eval_out:
    for spec in source_specs:
        part_dir = REFRESH_ROOT / spec['name']
        for line in (part_dir / 'train_dataset.jsonl').read_text(encoding='utf-8').splitlines():
            if line and line not in seen_train:
                seen_train.add(line)
                train_out.write(line + '\n')
                train_rows += 1
        for line in (part_dir / 'eval_dataset.jsonl').read_text(encoding='utf-8').splitlines():
            if line and line not in seen_eval:
                seen_eval.add(line)
                eval_out.write(line + '\n')
                eval_rows += 1

DATA_REFRESH_SUMMARY = {
    'refresh_tag': REFRESH_TAG,
    'sources': source_specs,
    'merged_dir': str(MERGED_REFRESH_DIR),
    'train_rows': train_rows,
    'eval_rows': eval_rows,
}

print('\nDATA_REFRESH_SUMMARY_JSON')
print(json.dumps(DATA_REFRESH_SUMMARY, indent=2))

Running: uv run tcrb run --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/ambiguous_ops_leak_probe.json --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-source-hard

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 126.43it/s]
Planner: hf_qwen2_5_3b_base
Wrote benchmark results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-source-hard/result.json
Wrote markdown summary: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-source-hard/summary.md
Running: uv run tcrb finetune-data --input-json /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-source-hard/result.json --output-dir /kaggle/working/tool-calling-reliability-benchmark

In [48]:
print('DATA_REFRESH_SUMMARY_JSON')
print(json.dumps(DATA_REFRESH_SUMMARY, indent=2))

DATA_REFRESH_SUMMARY_JSON
{
  "refresh_tag": "hard-refresh-v1",
  "sources": [
    {
      "name": "hard",
      "workload": "workloads/enriched/ambiguous_ops_leak_probe.json",
      "label": "toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-source-hard",
      "split_seed": "111"
    },
    {
      "name": "baseline",
      "workload": "workloads/enriched/customer_support.json",
      "label": "toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-source-baseline",
      "split_seed": "112"
    }
  ],
  "merged_dir": "/kaggle/working/tool-calling-reliability-benchmark/finetuned-models/hard-refresh-v1/merged",
  "train_rows": 83,
  "eval_rows": 21
}


In [ ]:
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
MAX_STEPS = 90
HF_TOKEN_LOCAL = str(globals().get('HF_TOKEN', os.environ.get('HF_TOKEN', ''))).strip()
if not HF_TOKEN_LOCAL:
    raise RuntimeError('HF_TOKEN is missing in kernel environment.')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this retrain step.')

print('[retrain] GPU:', torch.cuda.get_device_name(0))
print('[retrain] train file:', REFRESH_TRAIN_JSONL)
print('[retrain] eval file:', REFRESH_EVAL_JSONL)

train_ds = load_dataset('json', data_files=str(REFRESH_TRAIN_JSONL), split='train')
eval_ds = load_dataset('json', data_files=str(REFRESH_EVAL_JSONL), split='train')
print(f"[retrain] rows train={len(train_ds)} eval={len(eval_ds)}")


def _format_row(row):
    prompt = json.dumps(row['prompt'], sort_keys=True)
    completion = json.dumps(row['completion'], sort_keys=True)
    return {'text': f'Prompt: {prompt}\nCompletion: {completion}'}


train_ds = train_ds.map(_format_row, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(_format_row, remove_columns=eval_ds.column_names)

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN_LOCAL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN_LOCAL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

sft_config = SFTConfig(
    output_dir='outputs/ft-notebook/retrain-workdir',
    dataset_text_field='text',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    logging_steps=5,
    max_length=512,
    report_to='none',
    bf16=bool(torch.cuda.is_bf16_supported()),
    fp16=not bool(torch.cuda.is_bf16_supported()),
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
)

print(f'[retrain] Starting training for max_steps={MAX_STEPS}')
trainer.train()

final_dir = REPO_ROOT / 'outputs' / 'ft-notebook' / 'final'
backup_dir = REPO_ROOT / 'outputs' / 'ft-notebook' / f"backup-before-{REFRESH_TAG}-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"
if final_dir.exists():
    backup_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(final_dir, backup_dir)

trainer.save_model(str(final_dir))

RETRAIN_SUMMARY = {
    'model_id': MODEL_ID,
    'max_steps': MAX_STEPS,
    'train_rows': len(train_ds),
    'eval_rows': len(eval_ds),
    'final_dir': str(final_dir),
    'backup_dir': str(backup_dir) if backup_dir.exists() else None,
}

print('\nRETRAIN_SUMMARY_JSON')
print(json.dumps(RETRAIN_SUMMARY, indent=2))

[retrain] GPU: Tesla T4
[retrain] train file: /kaggle/working/tool-calling-reliability-benchmark/finetuned-models/hard-refresh-v1/merged/train_dataset.jsonl
[retrain] eval file: /kaggle/working/tool-calling-reliability-benchmark/finetuned-models/hard-refresh-v1/merged/eval_dataset.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

[retrain] rows train=83 eval=21


Map:   0%|          | 0/83 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


[retrain] Starting training for max_steps=90


Step,Training Loss
5,1.994061
10,1.155490
15,0.450186
20,0.239952
25,0.173173
30,0.119965
35,0.085132
40,0.052163
45,0.048928
50,0.039525


/tmp/ipykernel_80/1299509271.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  backup_dir = REPO_ROOT / 'outputs' / 'ft-notebook' / f"backup-before-{REFRESH_TAG}-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"



RETRAIN_SUMMARY_JSON
{
  "model_id": "Qwen/Qwen2.5-3B-Instruct",
  "max_steps": 90,
  "train_rows": 83,
  "eval_rows": 21,
  "final_dir": "/kaggle/working/tool-calling-reliability-benchmark/outputs/ft-notebook/final",
  "backup_dir": "/kaggle/working/tool-calling-reliability-benchmark/outputs/ft-notebook/backup-before-hard-refresh-v1-20260403-125809"
}


In [50]:
import subprocess
import sys

deps_cmd = [
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'trl',
    'transformers',
    'peft',
    'datasets',
    'accelerate',
    'bitsandbytes',
]
print('Installing deps:', ' '.join(deps_cmd))
res = subprocess.run(deps_cmd, text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'dep install failed with code {res.returncode}')
print('Dependency install complete.')

Installing deps: /usr/bin/python3 -m pip install -q trl transformers peft datasets accelerate bitsandbytes
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00

Dependency install complete.


In [52]:
import json
import subprocess

POST_LABEL_PREFIX = f"{FAST_LABEL_PREFIX}-{REFRESH_TAG}-posttrain"


def run_ms_post(workload: str, planner_cfg: str, label: str) -> None:
    cmd = [
        'uv', 'run', 'tcrb', 'multi-seed',
        '--config', str(FAST_CONFIG),
        '--workload', workload,
        '--seeds', FAST_SEEDS,
        '--planner-config', planner_cfg,
        '--label', label,
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'post-retrain multi-seed failed for {label} with code {rc}')


def load_ms_post(label: str) -> dict:
    payload_path = REPO_ROOT / 'runs' / label / 'multi_seed.json'
    payload = json.loads(payload_path.read_text(encoding='utf-8'))
    return {r['policy']: r['metrics'] for r in payload.get('aggregate_policy_metrics', [])}


post_base_label = f'{POST_LABEL_PREFIX}-base'
post_ft_label = f'{POST_LABEL_PREFIX}-ft'
post_null_label = f'{POST_LABEL_PREFIX}-null'

run_ms_post(FAST_WORKLOAD, BASE_PLANNER, post_base_label)
run_ms_post(FAST_WORKLOAD, FT_PLANNER, post_ft_label)
run_ms_post(FAST_WORKLOAD, str(FAST_NULL_PLANNER.relative_to(REPO_ROOT)), post_null_label)

post_base = load_ms_post(post_base_label)
post_ft = load_ms_post(post_ft_label)
post_null = load_ms_post(post_null_label)
post_policies = sorted(set(post_base) & set(post_ft) & set(post_null))

post_ft_s = [post_ft[p]['task_success_rate']['mean'] - post_base[p]['task_success_rate']['mean'] for p in post_policies]
post_ft_i = [post_ft[p]['invalid_tool_call_rate']['mean'] - post_base[p]['invalid_tool_call_rate']['mean'] for p in post_policies]
post_null_s = [post_null[p]['task_success_rate']['mean'] - post_base[p]['task_success_rate']['mean'] for p in post_policies]
post_null_i = [post_null[p]['invalid_tool_call_rate']['mean'] - post_base[p]['invalid_tool_call_rate']['mean'] for p in post_policies]

post_ft_success_delta = sum(post_ft_s) / len(post_ft_s) if post_ft_s else 0.0
post_ft_invalid_delta = sum(post_ft_i) / len(post_ft_i) if post_ft_i else 0.0
post_null_success_delta = sum(post_null_s) / len(post_null_s) if post_null_s else 0.0
post_null_invalid_delta = sum(post_null_i) / len(post_null_i) if post_null_i else 0.0

post_adapter_adv_success = post_ft_success_delta - post_null_success_delta
post_adapter_adv_invalid = post_ft_invalid_delta - post_null_invalid_delta
post_nonflat = max(
    abs(post_ft_success_delta),
    abs(post_ft_invalid_delta),
    abs(post_adapter_adv_success),
    abs(post_adapter_adv_invalid),
) > 1e-4

post_signal_pass = (
    (post_ft_success_delta > 0.0)
    and (post_ft_invalid_delta <= 0.0)
    and (post_adapter_adv_success > 0.003)
    and (post_adapter_adv_invalid <= 0.0)
    and post_nonflat
)

POST_FAST_TRIAD_SUMMARY = {
    'label_prefix': POST_LABEL_PREFIX,
    'workload': FAST_WORKLOAD,
    'seeds': FAST_SEEDS,
    'policies': post_policies,
    'ft_base_success_delta': post_ft_success_delta,
    'ft_base_invalid_delta': post_ft_invalid_delta,
    'null_base_success_delta': post_null_success_delta,
    'null_base_invalid_delta': post_null_invalid_delta,
    'adapter_adv_success': post_adapter_adv_success,
    'adapter_adv_invalid': post_adapter_adv_invalid,
    'nonflat': post_nonflat,
    'fast_signal_pass': post_signal_pass,
}

BEFORE_AFTER_FAST_TRIAD = {
    'before': FAST_TRIAD_SUMMARY,
    'after': POST_FAST_TRIAD_SUMMARY,
    'delta': {
        'ft_base_success_delta': POST_FAST_TRIAD_SUMMARY['ft_base_success_delta'] - FAST_TRIAD_SUMMARY['ft_base_success_delta'],
        'ft_base_invalid_delta': POST_FAST_TRIAD_SUMMARY['ft_base_invalid_delta'] - FAST_TRIAD_SUMMARY['ft_base_invalid_delta'],
        'adapter_adv_success': POST_FAST_TRIAD_SUMMARY['adapter_adv_success'] - FAST_TRIAD_SUMMARY['adapter_adv_success'],
        'adapter_adv_invalid': POST_FAST_TRIAD_SUMMARY['adapter_adv_invalid'] - FAST_TRIAD_SUMMARY['adapter_adv_invalid'],
    },
}

if POST_FAST_TRIAD_SUMMARY['fast_signal_pass']:
    POST_RETRAIN_DECISION = 'GO-PHASE-2: post-retrain fast triad found meaningful signal.'
else:
    POST_RETRAIN_DECISION = 'NO-LIFT: post-retrain fast triad still weak/flat.'

print('\nPOST_FAST_TRIAD_SUMMARY_JSON')
print(json.dumps(POST_FAST_TRIAD_SUMMARY, indent=2))
print('\nBEFORE_AFTER_FAST_TRIAD_JSON')
print(json.dumps(BEFORE_AFTER_FAST_TRIAD, indent=2))
print('\nPOST_RETRAIN_DECISION =', POST_RETRAIN_DECISION)

Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-posttrain-base

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 125.47it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-posttrain-base/multi_seed.json
Wrote multi-seed summary: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-posttrain-base/multi_seed_summary.md
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_fast_signal_cs_v1.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --planner-config co

In [53]:
print('POST_FAST_TRIAD_SUMMARY_JSON')
print(json.dumps(POST_FAST_TRIAD_SUMMARY, indent=2))
print('BEFORE_AFTER_FAST_TRIAD_JSON')
print(json.dumps(BEFORE_AFTER_FAST_TRIAD, indent=2))
print('POST_RETRAIN_DECISION =', POST_RETRAIN_DECISION)

POST_FAST_TRIAD_SUMMARY_JSON
{
  "label_prefix": "toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel-hard-refresh-v1-posttrain",
  "workload": "workloads/enriched/customer_support.json",
  "seeds": "11,22,33",
  "policies": [
    "exponential_backoff_jitter",
    "naive_retry"
  ],
  "ft_base_success_delta": 0.0,
  "ft_base_invalid_delta": 0.0,
  "null_base_success_delta": 0.0,
  "null_base_invalid_delta": 0.0,
  "adapter_adv_success": 0.0,
  "adapter_adv_invalid": 0.0,
  "nonflat": false,
  "fast_signal_pass": false
}
BEFORE_AFTER_FAST_TRIAD_JSON
{
  "before": {
    "label_prefix": "toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-fastsig-cs-v1-kernel",
    "workload": "workloads/enriched/customer_support.json",
    "seeds": "11,22,33",
    "policies": [
      "exponential_backoff_jitter",
      "naive_retry"
    ],
    "ft_base_success_delta": 0.0,
    "ft_base_invalid_delta": 0.0,
    "null_base_success_delta": 0.0,
    "null_base_invalid_delta": 0.0,
    "adapter_adv_s